## Variant region list:
- load the file of all sequences/oligo names (80215)
- make a list of these with a column for the name, sequence, type (if variant: ALT, if reference: REF, if region: region, control (true/false), gene association (all information about the gene (i.e. name with | and ~RAS ...))
- example table is shown [here](https://docs.google.com/spreadsheets/d/1YfepW_nv14024v8KwveGaJTbcBX6pRKRUohl7jZhltY/edit#gid=0)
### Questions:
- Some rows have ALT_, REF_, what does it mean if it has no REF_ and ALT_ is it a region then?
  - (1) id (not too long), (2) sequence, (3) category,  (4) class (test, variant (pos/neg) control), (5) source ? for our design ("candidate CRE nearby 536 cardiac, neuro, cava and random genes"), (6) ref_sequence ? CLEA controlls? what with the sequences (general controlls we took them actually from hg19, I think?,  (7) chrom (NA possible), (8) chrom_start (NA possible), (9) chrom_end (NA possible),  (10) variant_class (NA possible), (11) variant_pos (NA possible), (12) SPDI (NA possible), (13) allele (NA possible),  (14) info (free form)

### Process:
- We identified regions near TSS of genes (we looked for variants within these regions (centered in these regions (100bp from the center)))
- Total number of different regions: 80215
- First table:
  - for each row: one header with the informations it has ()


In [1]:
# imports 
import pandas as pd 
from Bio import SeqIO
# use the config file in yaml format
import yaml
import os
import re
import gzip # gzipped files
import hashlib

# read config
config_path = "/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/05_variant_region_list/config/config.yaml"
config_path = "/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/global80K_config.yaml"
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/global80K_config.yaml"
with open(config_path, 'r') as ymlfile:
    config = yaml.safe_load(ymlfile)

In [2]:
# load the data
design_fasta = '/fast/groups/ag_kircher/MPRA/IGVF_Y1_design/resources/association_data/design_no_duplicates_sequence_and_header.fa'
design_fasta = 'resources/design_no_duplicates_sequence_and_header.fa'
design_fasta = config['files']['final_design']['design_fasta']

# read the fasta file with the sequences and prepare a tsv with header and sequence using biopython
records = list(SeqIO.parse(design_fasta, "fasta"))
design_df = pd.DataFrame(columns=['header', 'sequence'])
header = [] 
sequence = []
for record in records:
    header.append(record.id)
    sequence.append(str(record.seq))

design_df['header'] = header
design_df['sequence'] = sequence

# label is the string in front of the first ":"
design_df['label'] = design_df['header'].str.split(':').str[0]
cardiac_neuro_cava_random = design_df[design_df['label'] == 'cardiac_neuro_cava_random']

design_df

,header,sequence,label
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,cardiac_neuro_cava_random
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,cardiac_neuro_cava_random
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,cardiac_neuro_cava_random
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,cardiac_neuro_cava_random
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,cardiac_neuro_cava_random
...,...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,MK
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,MK
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,MK
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,MK


### Trying to find unmerged and merged headers (does not work properly)

In [3]:
# read unmerged file
unmerged_file = config['files']['final_design']['design_with_duplicates']
unmerged_file = "/home/kisa/coding/80K_MPRA/80K-Analysis/05_variant_region_list/resources/design.fa"

# if file is gzipped, use the following
if unmerged_file.endswith(".gz"):
    file_handler = gzip.open(unmerged_file, "rt")
else:
    file_handler = open(unmerged_file, "r")

records = list(SeqIO.parse(file_handler, "fasta"))
unmer_design_df = pd.DataFrame(columns=['header', 'sequence'])
header = [] 
sequence = []
for record in records:
    header.append(record.id)
    sequence.append(str(record.seq))

unmer_design_df['header'] = header
unmer_design_df['sequence'] = sequence
unmer_design_df["unmerged"] = True
unmer_design_df = unmer_design_df[['header', 'unmerged']]
unmer_design_df.head()

,header,unmerged
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,True
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,True
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,True
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,True
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,True


In [4]:
# merge unmer_design_df and design_df on header
# the lines without label are merged
complete_design_df = design_df.merge(unmer_design_df, on="header", how="left")
complete_design_df
# number of na values in "label"
merged_header_df = complete_design_df[complete_design_df.unmerged.isna()] #3278
merged_header_df.shape
merged_header_list = merged_header_df.header.to_list()
merged_header_list[100]
# example 'cardiac_neuro_cava_random:REF_CSDE1|ENSG00000009307.17|EH38E1378368~NRAS|ENSG00000213281.5|EH38E1378368_rev_tile1-1'

'cardiac_neuro_cava_random:REF_CSDE1|ENSG00000009307.17|EH38E1378368~NRAS|ENSG00000213281.5|EH38E1378368_rev_tile1-1'

In [5]:
merged_header_df

,header,sequence,label,unmerged
708,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTAAGGAAGGGAGGGAGGGAGGGAGCGATCCCT...,cardiac_neuro_cava_random,NaN
709,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTGGACCACCTCCACCAACTGTCAGCTCACATC...,cardiac_neuro_cava_random,NaN
710,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTACTGTCCTTCCTGAGGCCTCCAGCATTATTG...,cardiac_neuro_cava_random,NaN
711,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTTTACTTATTTATTTATTTTTTGAGACAGGGT...,cardiac_neuro_cava_random,NaN
712,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTACCCCATAGTATGCCTCCCTCCCTCTCTCCT...,cardiac_neuro_cava_random,NaN
...,...,...,...,...
77523,C_positive_heart_AB:FLNA-ENST00000420627.5:Oli...,AGGACCGGATCAACTGGTCGCCCATTCCCAAGCTCCCACCTTGACG...,C_positive_heart_AB,NaN
77524,C_positive_heart_AB:FLNA-ENST00000420627.5:Oli...,AGGACCGGATCAACTGGGCCCAACCAAGGAACCTGGCCTGGTCTCA...,C_positive_heart_AB,NaN
77525,C_positive_heart_AB:FLNA-ENST00000420627.5:Oli...,AGGACCGGATCAACTCCACTCGCCCTGAGTCCACACAAGTTCCTGG...,C_positive_heart_AB,NaN
77526,C_positive_heart_AB:FLNA-ENST00000360319.9:Oli...,AGGACCGGATCAACTGTAAAATTGCCCAGGAGCCCGGGACGGGTGC...,C_positive_heart_AB,NaN


In [6]:
# count number of rows per label
# split based on "|" number in order to find pattern
merged_header_df["pipe_count"] = merged_header_df['header'].str.count(r'\|')
merged_header_df_cardiac = merged_header_df[merged_header_df["label"] == "cardiac_neuro_cava_random"]
merged_header_df_cardiac #1717 / 3278 => 52%
merged_header_df_cardiac.pipe_count.value_counts()
# pipe_count
# 8     596
# 7     471
# 4     341
# 10    309

/tmp/ipykernel_59271/2263432361.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_header_df["pipe_count"] = merged_header_df['header'].str.count(r'\|')


pipe_count
8     596
7     471
4     341
10    309
Name: count, dtype: int64

### Split the number of headers from cardiac_neuro_cava_random label by the number of pipes to find patterns
- found patterns
  - alternate oligos: ALT_ (one snp each)
  - reference oligos: REF_ (some are merged) also regions
  - regions: no REF_ or ALT_ (some are merged)

#### Split of cardiac_neuro_cava_random by number of pipes:
- pipe_count
- 8     596
- 7     471
- 4     341
- 10    309

##### 8 pipes: (596)
- all alt
- do all have same number of "_" 
- do all have same number and position of "~"
- do all end with a variant
- example: # 'cardiac_neuro_cava_random:ALT_DRD4|ENSG00000069696.7|EH38E2937745_fwd_tile1-1_DEAF1|ENSG00000177030.19|EH38E2937745|11-596480-T-C~DRD4|ENSG00000069696.7|EH38E2937745|11-596480-T-C'

In [7]:
# 8: 596 rows
cardiac_8_pipe_df = merged_header_df_cardiac[merged_header_df_cardiac["pipe_count"] == 8]
cardiac_8_pipe_df # 596

# count number of "_"
cardiac_8_pipe_df.header.str.count(r"_")
# count number and look at position of "~"

# end pattern matches variant patter?
cardiac_8_pipe_list = cardiac_8_pipe_df["header"].to_list()
cardiac_8_pipe_list[0]


'cardiac_neuro_cava_random:ALT_DRD4|ENSG00000069696.7|EH38E2937745_fwd_tile1-1_DEAF1|ENSG00000177030.19|EH38E2937745|11-596480-T-C~DRD4|ENSG00000069696.7|EH38E2937745|11-596480-T-C'

#### Short cut ideas:
- filter all header with variants: |(last pipe)<num>-<num>
  - Pattern ? if variant is there there is ALT in the beginning
- Find variant pattern -> its variant
- Find reference pattern -> its reference
- Find nothing -> its region
- check the numbers
  - Number control sequences: 6275
  - Number tested sequences: 73940
```
-------Design Number Summary (for cardiac_neuro_cava_random group) --------
Number of references: 18582
Number of alternative sequences: 46458
Number of regions (without variants): 8900
```
- contorls:
  - looking for variant controls (especially positive ones)
    - control labels with "ALT" in header: `cat /home/kisa/coding/80K_MPRA/80K-Analysis/05_variant_region_list/controll_header.tsv | grep -v "C_positive_neuron_CD"| grep "ALT" | awk -F ':' '{print $1}' | sort | uniq `
      - C_positive_heart_CAD
      - GC_Atrial_fib
      - GC_Kircher
      - GC_Liang
      - GC_Mendelian_variants
      - GC_Mohlke
      - GC_Selvarajan
      - MK
  - numbers of "ALT" in control header: 656 `cat /home/kisa/coding/80K_MPRA/80K-Analysis/05_variant_region_list/controll_header.tsv | grep -v "C_positive_neuron_CD"| grep "ALT" | wc -l` (removed `C_positive_neuron_CD` because these are element controls)



In [8]:
# helpful functions for checking reference, variants and regions
def check_variant(header, check_controls=False):
    """Checks if a header is a variant header"""
    # get the last part of the header
    last_part = header.split('|')[-1]
    # check if the last part if it matches the regex [\d]+-[\d]+
    if re.match(r'[\dA-Z]+-[\d]+', last_part): # is sufficient, because all these headers have ALT in their name
        return True
    else:
        return False

def check_reference(header):
    """Checks if a header is a reference header"""
    # check after the label if REF_ is in the header
    non_label_header = header.split(':')[1]
    if "REF_" in non_label_header.split('|')[0]:
        return True
    else:
        return False

def check_region(header):
    """
    Checks if a header is a region header
    A region header is here defined as a header without ALT_ or REF_ after the first ":" 
    """
    # check after the label if REF_ is in the header
    non_label_header = header.split(':')[1]
    if "REF_" in non_label_header.split('|')[0]:
        return False
    elif "ALT_" in non_label_header.split('|')[0]:
        return False
    else:
        return True

def check_region_variant_reference_numbers(header_list, check_controls=False):
    """Checks if a header is a variant, reference or region header"""
    ref_counter = 0
    var_counter = 0
    region_counter = 0
    unknown_counter = 0
    for header in header_list:
        if header.split(':')[0] != 'cardiac_neuro_cava_random':
            continue
            # TODO: add way to check if sequence is a alternative in controls
            if check_controls:
                continue
        if check_variant(header):
            var_counter += 1
            header_type = 'ALT'
        elif check_reference(header):
                ref_counter += 1
                header_type = 'REF'
        elif check_region(header):    
            region_counter += 1
            header_type = 'region'
        else:
            header_tpye = 'unknown'
            unknown_counter += 1
            print(f'Found unknown header: {header}')
    return ref_counter, var_counter, region_counter

def create_fasta_df_from_one_line_sequence_fasta(fasta_path, filter_cardiac=False):
    """Creates a dataframe with header and sequence from a fasta file which has one line per sequence"""
    records = list(SeqIO.parse(fasta_path, "fasta"))
    design_df = pd.DataFrame(columns=['header', 'sequence'])
    header = [] 
    sequence = []
    for record in records:
        header.append(record.id)
        sequence.append(str(record.seq))

    design_df['header'] = header
    design_df['sequence'] = sequence

    if filter_cardiac:
        design_df['label'] = design_df['header'].str.split(':').str[0]
        design_df = design_df[design_df['label'] == 'cardiac_neuro_cava_random']

    return design_df

def get_gene_name_from_header(header):
    """Identify the gene name (e.g. MYH6) from the header: sequence after first ":" and before first "|" then crop the sequence after the first "_" if it exists"""
    gene_name = header.split(':')[1].split('|')[0]
    if '_' in gene_name:
        gene_name = gene_name.split('_')[1]
    return gene_name

def write_variants_fasta(design_fasta_path, fasta_out_directory):
    """Write all variants to a fasta file with the header and sequence"""
    design_df = create_fasta_df_from_one_line_sequence_fasta(design_fasta_path, filter_cardiac=False)
    design_df['is_variant'] = design_df['header'].apply(check_variant)
    design_df = design_df[design_df['is_variant'] == True]
    # write the fasta file
    output_path = os.path.join(fasta_out_directory, f'identified_variants_{design_df.shape[0]}.fa')
    with open(output_path, 'w') as f:
        for index, row in design_df.iterrows():
            f.write('>' + row['header'] + '\n' + row['sequence'] + '\n')
    return design_df

def write_region_fasta(design_fasta_path, fasta_out_directory):
    """Write all regions to a fasta file with the header and sequence"""
    design_df = create_fasta_df_from_one_line_sequence_fasta(design_fasta_path, filter_cardiac=True)
    design_df['is_region'] = design_df['header'].apply(check_region)
    design_df = design_df[design_df['is_region'] == True]
    # write the fasta file
    output_path = os.path.join(fasta_out_directory, f'identified_regions_{design_df.shape[0]}.fa')
    with open(output_path, 'w') as f:
        for index, row in design_df.iterrows():
            f.write('>' + row['header'] + '\n' + row['sequence'] + '\n')
    return design_df

def write_reference_fasta(design_fasta_path, fasta_out_directory):
    """Write all references to a fasta file with the header and sequence"""
    design_df = create_fasta_df_from_one_line_sequence_fasta(design_fasta_path, filter_cardiac=True)
    design_df['is_reference'] = design_df['header'].apply(check_reference)
    design_df = design_df[design_df['is_reference'] == True]
    # write the fasta file
    output_path = os.path.join(fasta_out_directory, f'identified_references_{design_df.shape[0]}.fa')
    with open(output_path, 'w') as f:
        for index, row in design_df.iterrows():
            f.write('>' + row['header'] + '\n' + row['sequence'] + '\n')
    return design_df



In [9]:
# iterate all headers and get the gene name of interest
gene_names = set()
not_cardiac = 0
header_list = design_df['header'].to_list()
for hdr in header_list:
    if hdr.split(':')[0] != 'cardiac_neuro_cava_random':
        not_cardiac += 1
        continue
    # put gene name into set
    gene_name = get_gene_name_from_header(hdr)
    gene_names.add(gene_name)

# check the number of the gene_name set
print(f'Number of unique "assiciated" genes: {len(gene_names)}')  # => Found all (same number as in summary presentation) 525 "associated" genes

Number of unique "assiciated" genes: 525


In [10]:
print('Number control sequences: %s'%(not_cardiac)) # 6275

num_cardiac = design_df.shape[0] - not_cardiac # 80215 - 6275 => 73940 (can be checked with grep and are the correct numbers)
print('Number tested sequences: %s'%(num_cardiac))


Number control sequences: 6275
Number tested sequences: 73940


In [11]:

ref_counter, var_counter, region_counter = check_region_variant_reference_numbers(header_list)

if ref_counter + var_counter + region_counter != num_cardiac:
    raise ValueError("The numbers do not match")
else:
    print("The numbers of tested sequences and region_variant_reference function match")

print('-------Design Number Summary --------\nNumber of references: %s\nNumber of alternative sequences: %s\nNumber of regions (without variants): %s'%(ref_counter, var_counter, region_counter)) # 02.04.2024: 18582 + 46458 + 8900 = 73940

# we want: 28000 cCREs we got 8900
design_fasta_file = config['files']['final_design']['design_fasta']
output_fasta_directory = 'resources/'
# write_variants_fasta(design_fasta_file, output_fasta_directory)
# write_reference_fasta(design_fasta_file, output_fasta_directory)
# write_region_fasta(design_fasta_file, output_fasta_directory)
# Problem: we are not sure about the exact numbers and the formats might be different
  # - how do I check if the number of variants (currently 46458) is correct?

The numbers of tested sequences and region_variant_reference function match
-------Design Number Summary --------
Number of references: 18582
Number of alternative sequences: 46458
Number of regions (without variants): 8900


In [12]:
# read all headers and check for the regex pattern of variant info after the last pipe ("|")
# count the number of found variant patterns
# if the pattern is found, check if it has ALT_ after the first ":"
# count the number of found variant pattersn with ALT_
var_count = 0
var_count_alt = 0
for hdr in header_list:
    if hdr.split(':')[0] != 'cardiac_neuro_cava_random':
        continue
    if re.match('ALT_', hdr.split(':')[1]):
        # if it matches, count it
        var_count_alt += 1
        # print(hdr)
        # break
        # get the last part of the header
        last_part = hdr.split('|')[-1]
        # check if the last part if it matches the regex [\d]+-[\d]+
        if re.match(r'[\dA-Z]+-[\d]+', last_part):
            var_count += 1
        else:
            print(hdr)
        
# check if the number of variants is the same as the number of variants with ALT_
if var_count == var_count_alt:
    print("All variants have ALT_ in the header")

All variants have ALT_ in the header


In [13]:
# list of all unique labels
unique_labels = design_df['label'].unique().tolist()
unique_labels

['cardiac_neuro_cava_random',
 'GC_Atrial_fib',
 'GC_Liang',
 'GC_Selvarajan',
 'GC_Mohlke',
 'GC_Kircher',
 'GC_Mendelian_variants',
 'C_positive_heart_CAD',
 'GC_Cort_Chengyu',
 'GC_GABA_Chengyu',
 'GC_Glut_Chengyu',
 'GC_Hon',
 'GC_Vista',
 'GC_DNase_positive',
 'GC_DNase_negative_brain',
 'GC_DNase_negative_blood',
 'C_negative_heart_MK',
 'C_negative_neuron_MK',
 'C_negative_neuron_NP',
 'C_positive_heart_MK',
 'C_positive_neuron_CD',
 'C_positive_neuron_MK',
 'C_positive_neuron_NP',
 'C_positive_heart_AB',
 'C_SLEA',
 'GC_DNase_positive_shuffeled',
 'GC_DNase_negative_brain_shuffeled',
 'GC_DNase_negative_blood_shuffeled',
 'MK']

In [14]:
# investigate the variants
for i in range(40, 48):
    print(design_df['header'].tolist()[i])


cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779571_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779574_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779580_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779583_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779643_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779653_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779655_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779714_fwd_tile1-1


In [15]:
for lable in unique_labels:
    if lable == 'cardiac_neuro_cava_random':
        continue
    df = design_df[design_df['label'] == lable]
    print(lable)
    print(df['header'].str.count('\|').value_counts())

GC_Atrial_fib
header
2    34
4    11
Name: count, dtype: int64
GC_Liang
header
1    16
Name: count, dtype: int64
GC_Selvarajan
header
1    241
2    105
4     10
3      8
Name: count, dtype: int64
GC_Mohlke
header
4     26
12     5
8      3
Name: count, dtype: int64
GC_Kircher
header
400    203
Name: count, dtype: int64
GC_Mendelian_variants
header
2    161
1     48
Name: count, dtype: int64
C_positive_heart_CAD
header
0    97
Name: count, dtype: int64
GC_Cort_Chengyu
header
3    184
2      1
Name: count, dtype: int64
GC_GABA_Chengyu
header
3    85
Name: count, dtype: int64
GC_Glut_Chengyu
header
3    40
Name: count, dtype: int64
GC_Hon
header
1    6
Name: count, dtype: int64
GC_Vista
header
1    256
Name: count, dtype: int64
GC_DNase_positive
header
0    41
Name: count, dtype: int64
GC_DNase_negative_brain
header
0    15
Name: count, dtype: int64
GC_DNase_negative_blood
header
0    15
Name: count, dtype: int64
C_negative_heart_MK
header
0    243
Name: count, dtype: int64
C_negative_neu

### Metadata file: [document](https://docs.google.com/document/d/1ThHgLjMnS2r-vv_4ZHHO9y4K1Qbc2S2WKKUDw__iYXU/edit)
- I want to have a table of id, sequence, category, class, source, ref_sequence, chrom, chrom_start, chrom_end, variant_class, variant_pos, SPDI, allele, info

In [16]:
import os
import pandas as pd
import numpy as np
focusing_label = 'cardiac_neuro_cava_random'


def get_category(header):
    """Get the category of the header
    if C_SLEA in tmp_label => synthetic
    elif scramble in header => scrambled
    elif ref_ or alt_ in header => variant 
    else element
    """
    label = get_label(header)
    if label in synthetic_control_groups:
        return 'synthetic'
    elif 'scramble' in header:
        return 'scrambled'
    elif ref_or_alt_in_header(header):
        return 'variant'
    else:
        return 'element'


def get_class(header):
    """
    Get the class of the header
    A class is according to the IGVF metadata format: test, variant positive control, variant negative control, 
          element active control or element inactive control
    """
    # known pattern for cardiac_neuro_cava_random: all are "test"
    if 'cardiac_neuro_cava_random' in header:
        return 'test'
    elif is_positive_control(header):
        return get_positive_class(header)
    elif is_negative_control(header):
        return get_negative_class(header)
    else:
        return 'NA'
        
    
    
def get_source(header):
    """Get the source of the header"""
    # only known pattern: for cardiac_neuro_cava_random: "candidate CRE nearby 536 cardiac, neuro, cava and random genes"
    if 'cardiac_neuro_cava_random' in header:
        return 'candidate CRE nearby cardiac, neuro, cava and random genes'
    if 'GC_' in header:
        label = header.split(':')[0]
        return 'IGVF general controls (%s)'%(label)
    else:
        return 'NA'
    
    
def create_path(path):
    """Check if path exists if not create it"""
    dir_path = os.path.dirname(path)
    if not os.path.exists(dir_path):
        os.makedirs(dir_path)
        
        
def create_fasta_df_from_one_line_sequence_fasta(fasta_path, filter_cardiac=False):
    """Creates a dataframe with header and sequence from a fasta file which has one line per sequence"""
    records = list(SeqIO.parse(fasta_path, "fasta"))
    design_df = pd.DataFrame(columns=['header', 'sequence'])
    header = [] 
    sequence = []
    for record in records:
        header.append(record.id)
        sequence.append(str(record.seq))

    design_df['header'] = header
    design_df['sequence'] = sequence

    if filter_cardiac:
        design_df['label'] = design_df['header'].str.split(':').str[0]
        design_df = design_df[design_df['label'] == 'cardiac_neuro_cava_random']

    return design_df


def get_info(header):
    """Set information: e.g. wrong coordinates used"""
    label = get_label(header)
    # wrong coordinates used
    if label in wrong_coordinates_groups:
        return '%s;wrong genome build (GRCh37) used for these sequences'%(label)
    else:
        return '%s'%(label)
    
def get_label(header):
    return header.split(':')[0]


def ref_or_alt_in_header(header):
    """Checks if "ref_" or "_alt" is in the header"""
    if 'ref_' in header.lower() or 'alt_' in header.lower():
        return True
    return False


def get_negative_class(header):
    if ref_or_alt_in_header(header):
        return 'variant negative control'
    return 'element inactive control'


def get_positive_class(header):
    if ref_or_alt_in_header(header):
        return 'variant positive control'
    return 'element active control'


def is_positive_control(header):
    """Checks if header is positive control"""
    label = get_label(header)
    if label in positive_control_groups:
        return True
    return False


def is_negative_control(header):
    """Checks if header is negative control"""
    label = get_label(header)
    if label in negative_control_groups:
        return True
    return False

def get_control_reference(row):
    """Get reference sequence for control"""
    if row['ref_sequence'] != 'NA':
        return row['ref_sequence']
    else:
        # Add additional specification if needed here
        return 'GRCh38'
    

def get_region_match_name(header):
    """Get the region match name from the header
    1. split by ':' and take the second part [1]
    2. split by '_' and take the second part [1]
    """
    if header.split(':')[0] == 'cardiac_neuro_cava_random':
        if 'ALT_' in header or 'REF_' in header:
            if '~' in header:
                # replace ~ to ,: cardiac_neuro_cava_random:REF_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1
                header = header.replace('~', ',')
            # easy types: 
            # cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1'
            # cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1'
            return ':'.join(header.split(':')[1:]).split('_')[1]
        else: # cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778480_fwd_tile1-1
            if '~' in header:
                # replace ~ to ,: cardiac_neuro_cava_random:APOL2|ENSG00000128335.14|EH38E3478577~APOL4|ENSG00000100336.18|EH38E3478577_rev_tile1-1
                header = header.replace('~', ',')
            return ':'.join(header.split(':')[1:]).split('_')[0]
    else:
        return header

# def get_control_category_from_class(row):
#     """Set category of controls from class"""
#     if row['tmp_label'] == focusing_label:
#         return get_category(row['header'])
    
#     category_conversion_dict = {
#         'variant positive control': 'variant', 
#         'variant negative control': 'variant', 
#         'element active control': 'element', 
#         'element inactive control': 'element'
#     }
    
#     # C_SLEA
#     if "C_SLEA" in row['tmp_label']:
#         return 'synthetic'
    
#     # scramble in name
#     if "scramble" in row['header']:
#         return 'scrambled'
    
#     return category_conversion_dict[row['class']]



In [17]:
# load fasta (header, sequence) and add columns with NA values 
# design_fasta = config['files']['reference']
design_fasta = config['files']['final_design']['design_fasta']
pre_metadata_df = create_fasta_df_from_one_line_sequence_fasta(design_fasta, filter_cardiac=False)

# add temporary label column
pre_metadata_df['tmp_label'] = pre_metadata_df['header'].apply(get_label)

# which controls are considered positive and negative
controls = pre_metadata_df[pre_metadata_df['tmp_label'] != focusing_label]
# get all unique values of the tmp_label column 
all_control_groups = controls['tmp_label'].unique()

positive_control_groups = ['C_positive_neuron_NP', 'C_positive_neuron_MK', 'C_positive_neuron_CD']
synthetic_control_groups = ['C_SLEA']
wrong_coordinates_groups = [group for group in all_control_groups if "dnase" in group.lower()] # add information that these used wrong coordinates

negative_control_groups = [group for group in all_control_groups if not group in positive_control_groups]


# add empty columns name, category, class, source, ref_sequence, chrom, chrom_start, chrom_end, variant_class, variant_pos, SPDI, allele, info
pre_metadata_df['name'] = 'NA' # will be the header: later: genome coordinates and for shuffled,scrambled and synthetic a clear prefix
pre_metadata_df['category'] = 'NA' # for cardiac_neuro_cava_random: if "ALT_" or "REF_" in header, then "variant", else "element" otherwise leave NA
pre_metadata_df['class'] = 'NA' # if cardiac_neuro_cava_random: "test", else leave NA (TODO: find pattern in controls for variants and elements and positive and negative controls)
pre_metadata_df['source'] = 'NA' # if cardiac_neuro_cava_random: "candidate CRE nearby 536 cardiac, neuro, cava and random genes", if "GC" in header, then "IGVF general controls", else leave NA
# pre_metadata_df['ref_seq'] = 'NA' # will be added later if "cardiac_neuro_cava_random" then "hg38", if "CLEA" in header, then "hg18", else leave NA
# pre_metadata_df['seq_chr'] = 'NA' # leave NA (TODO: add this from the regions.bed file in the final_design/*/final_design directory)
# pre_metadata_df['seq_start'] = 'NA' # leave NA (TODO: see above)
# pre_metadata_df['seq_end'] = 'NA' # leave NA (TODO: see above)
pre_metadata_df['variant_class'] = 'NA' # if "cardiac_neuro_cava_random" and category is "variant" then "SNV", else leave NA
pre_metadata_df['variant_pos'] = 'NA' # leave NA (TODO: see above + compute from reference position (pos of variant - 1) - pos of reference = variant_pos (0-based))
pre_metadata_df['SPDI'] = 'NA' # leave NA
pre_metadata_df['allele'] = 'NA' # if cardiac_neuro_cava_random if REF_ in header then "ref", if ALT_ in header then "alt", else leave NA
pre_metadata_df['info'] = 'NA' # leave NA

# name:
# pre_metadata_df['name'] = pre_metadata_df['sequence'].apply(lambda x: 'oligo_' + hashlib.md5(x.encode()).hexdigest())
pre_metadata_df['name'] = pre_metadata_df['header']
# check for duplicates
pre_metadata_df['name'].duplicated().sum() # 0


# class:
pre_metadata_df['class'] = pre_metadata_df['header'].apply(get_class)

# category:
pre_metadata_df['category'] = pre_metadata_df['header'].apply(get_category)


# source:
pre_metadata_df['source'] = pre_metadata_df['header'].apply(get_source)

# ref_sequence:
pre_metadata_df['ref_seq'] = pre_metadata_df['header'].apply(lambda x: 'GRCh38' if 'cardiac_neuro_cava_random' in x else ('hg18' if 'SLEA' in x else 'GRCh38'))

# leave NA for seq_chr, seq_start, seq_end (add later by loading bed class-wise)



# variant_class:
pre_metadata_df['variant_class'] = pre_metadata_df.apply(lambda x: 'SNP' if x['category'] == 'variant' else 'NA', axis=1)

# leave NA for variant_pos, SPDI, 

# allele:
pre_metadata_df['allele'] = pre_metadata_df['header'].apply(lambda x: 'ref' if 'REF_' in x else ('alt' if 'ALT_' in x else 'NA'))

# leave info as NA

pre_metadata_df['info'] = pre_metadata_df['header'].apply(get_info)
# pre_metadata_df


### Add variants to their reference
- read variant region map
- for each reference get the number of variants
- then get the variants coresponding to the reference

In [18]:
var_reg_map = '/home/kisa/coding/80K_MPRA/80K-Analysis/05_variant_region_list/resources/variant_region_map/variant_region_map.tsv.gz'

variant_region_map = pd.read_csv(var_reg_map, sep='\t')
print('Number of rows in variant region map: ', variant_region_map.shape[0])
print('Number of rows in variant region map of cardiac_neuro_cava_random: ', variant_region_map.loc[variant_region_map['Variant'].str.startswith('cardiac_neuro_cava_random')].shape[0])
print('Number of rows in variant region map of controls: ', variant_region_map.loc[~variant_region_map['Variant'].str.startswith('cardiac_neuro_cava_random')].shape[0])
# get number of unique REF_ID rows: 18883
ref_ids = variant_region_map['REF_ID'].unique()
len(ref_ids)
# get number of unique ALT_ID rows: 47044
alt_ids = variant_region_map['ALT_ID'].unique()
len(alt_ids)

## go through all references and get list of variants for each reference
tested_sequences_map = variant_region_map[variant_region_map['REF_ID'].str.startswith('cardiac_neuro_cava_random')]
tested_sequences_map.groupby('REF_ID')['ALT_ID'].apply(list).reset_index(name='alternatives')
tested_sequences_map.groupby('REF_ID')['ALT_ID'].apply(list).to_dict()

# get maximal number of alternatives per reference: 41 (cardiac_neuro_cava_random:REF_H1-3|ENSG00000124575.7|EH38E3697654_rev_tile1-1)
count_tbl = pd.DataFrame(tested_sequences_map.groupby('REF_ID')['ALT_ID'].count())
count_tbl[count_tbl['ALT_ID'] == 41]

Number of rows in variant region map:  47044
Number of rows in variant region map of cardiac_neuro_cava_random:  46374
Number of rows in variant region map of controls:  670


,ALT_ID
REF_ID,
cardiac_neuro_cava_random:REF_H1-3|ENSG00000124575.7|EH38E3697654_rev_tile1-1,41


In [19]:
variant_region_map.head()

,Variant,Region,REF_ID,ALT_ID
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...


#### Add seq_chrom, seq_strand, seq_start and seq_end to the table

In [20]:
# add chrom, seq_start, seq_end for tested sequences with bed file (/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/final_design/input/regions_5K.bed)

# load bed file
bed_file_test_seqs = pd.read_csv(config['files']['final_design']['test_seqs_region_file'], sep='\t', header=None)
bed_file_test_seqs.columns = ['seq_chr', 'seq_start', 'seq_end', 'tmp_bed_name', 'tmp_score', 'seq_strand']
data_types = {'seq_chr': str, 'seq_start': str, 'seq_end': str, 'tmp_bed_name': str, 'tmp_score': str, 'seq_strand': str}
bed_file_test_seqs = bed_file_test_seqs.astype(data_types) # seq_start + seq_end not int or float

# generate correct name for the pre_metadata_df (tmp_bed_name)
pre_metadata_df['tmp_bed_name'] = pre_metadata_df['header'].apply(get_region_match_name)

# all rows with cardiac_neuro_cava_random: 73940 all rows: 80215

# adding this information to pre_metadata_df (problem: wrong number of rows => 80215 (correct) vs 83751 (wrong)) 
pre_metadata_df = pre_metadata_df.merge(bed_file_test_seqs, on='tmp_bed_name', how='left')
# checking_chromposrefalt.fillna('NA', inplace=True)

pre_metadata_df['seq_chr'].isna().sum() # 49123 # 6275

# checking_chromposrefalt[checking_chromposrefalt['seq_chr'].isna()].tmp_label.value_counts() # 6275 


6275

##### Add for controls:

In [21]:
pre_metadata_df[pre_metadata_df['seq_chr'].isna()]['tmp_label'].value_counts()

tmp_label
MK                                   2397
C_positive_heart_AB                   909
GC_Selvarajan                         364
GC_Vista                              256
C_negative_heart_MK                   243
C_negative_neuron_MK                  222
C_negative_neuron_NP                  217
GC_Mendelian_variants                 209
GC_Kircher                            203
C_SLEA                                200
GC_Cort_Chengyu                       185
C_positive_neuron_NP                   99
C_positive_heart_CAD                   97
C_positive_heart_MK                    97
C_positive_neuron_MK                   96
C_positive_neuron_CD                   94
GC_GABA_Chengyu                        85
GC_DNase_positive_shuffeled            55
GC_Atrial_fib                          45
GC_DNase_positive                      41
GC_Glut_Chengyu                        40
GC_Mohlke                              34
GC_DNase_negative_blood_shuffeled      19
GC_Liang                

In [22]:
pre_metadata_df[pre_metadata_df['name'] == 'cardiac_neuro_cava_random:ALT_RIT1|ENSG00000143622.12|EH38E1387405_rev_tile1-1_RIT1|ENSG00000143622.12|EH38E1387405|1-155892969-T-C']

,header,sequence,tmp_label,name,category,class,source,variant_class,variant_pos,SPDI,allele,info,ref_seq,tmp_bed_name,seq_chr,seq_start,seq_end,tmp_score,seq_strand
31092,cardiac_neuro_cava_random:ALT_RIT1|ENSG0000014...,AGGACCGGATCAACTGATTTGGATTGAATGGTGGGAGAATGAGAAG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_RIT1|ENSG0000014...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",SNP,NA,NA,alt,cardiac_neuro_cava_random,GRCh38,RIT1|ENSG00000143622.12|EH38E1387405,chr1,155892803,155893150,.,-


In [23]:
pre_metadata_df[pre_metadata_df['class'] != 'test']

,header,sequence,tmp_label,name,category,class,source,variant_class,variant_pos,SPDI,allele,info,ref_seq,tmp_bed_name,seq_chr,seq_start,seq_end,tmp_score,seq_strand
73940,GC_Atrial_fib:rs7795510|CAV1|STARR-seq-AF~rs78...,AGGACCGGATCAACTTCATTTCATTATAATCAAAAAGGATTTTTAA...,GC_Atrial_fib,GC_Atrial_fib:rs7795510|CAV1|STARR-seq-AF~rs78...,element,element inactive control,IGVF general controls (GC_Atrial_fib),NA,NA,NA,NA,GC_Atrial_fib,GRCh38,GC_Atrial_fib:rs7795510|CAV1|STARR-seq-AF~rs78...,NaN,NaN,NaN,NaN,NaN
73941,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,AGGACCGGATCAACTCAGCTGCCCATGCTGGGACTGTGATTTTTTG...,GC_Atrial_fib,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,variant,variant negative control,IGVF general controls (GC_Atrial_fib),SNP,NA,NA,ref,GC_Atrial_fib,GRCh38,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,NaN,NaN,NaN,NaN,NaN
73942,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,AGGACCGGATCAACTAGAGCCCTTCTGGGGGCCCTGGCCACTGGCC...,GC_Atrial_fib,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,variant,variant negative control,IGVF general controls (GC_Atrial_fib),SNP,NA,NA,ref,GC_Atrial_fib,GRCh38,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,NaN,NaN,NaN,NaN,NaN
73943,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,AGGACCGGATCAACTAGGAAAGGCACTGGAAATTGTACTTACTCCA...,GC_Atrial_fib,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,variant,variant negative control,IGVF general controls (GC_Atrial_fib),SNP,NA,NA,ref,GC_Atrial_fib,GRCh38,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,NaN,NaN,NaN,NaN,NaN
73944,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,AGGACCGGATCAACTTTTGCAAAGGTATGGTTGGTGGATGGAGAAA...,GC_Atrial_fib,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,variant,variant negative control,IGVF general controls (GC_Atrial_fib),SNP,NA,NA,ref,GC_Atrial_fib,GRCh38,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,MK,MK:tile_2240|chr1-116244322+116244591|scramble...,scrambled,element inactive control,NA,NA,NA,NA,NA,MK,GRCh38,MK:tile_2240|chr1-116244322+116244591|scramble...,NaN,NaN,NaN,NaN,NaN
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,MK,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,scrambled,element inactive control,NA,NA,NA,NA,NA,MK,GRCh38,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,NaN,NaN,NaN,NaN,NaN
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,MK,MK:tile_18415|chr17-71181691+71181960|scramble...,scrambled,element inactive control,NA,NA,NA,NA,NA,MK,GRCh38,MK:tile_18415|chr17-71181691+71181960|scramble...,NaN,NaN,NaN,NaN,NaN
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,MK,MK:tile_14356|chr15-67031618+67031887|scramble...,scrambled,element inactive control,NA,NA,NA,NA,NA,MK,GRCh38,MK:tile_14356|chr15-67031618+67031887|scramble...,NaN,NaN,NaN,NaN,NaN


In [24]:
# get the name and header for the metadata file
name_header_match = pre_metadata_df[['name', 'header']]

name_header_match_path = config['files']['creating']['name_header_match'] 
# write the match table to a tsv
# create output path
create_path(name_header_match_path)
name_header_match.to_csv(name_header_match_path, sep='\t', index=False)

# order the columns as in the metadata file
pre_metadata_df_final = pre_metadata_df[['name', 'sequence', 'category', 'class', 'source', 'ref_seq', 'seq_chr', 'seq_start', 'seq_end', 'seq_strand', 'variant_class', 'variant_pos', 'SPDI', 'allele', 'info']]

pre_metadata_df_final

# # write the metadata file
metadata_path = config['files']['creating']['metadata_table']
# create_path(metadata_path)
# metadata_path = 'metadata_example_2703.tsv'
pre_metadata_df_final.to_csv(metadata_path, sep='\t', index=False)
# pre_metadata_df.to_csv(metadata_path, sep='\t', index=False)

# in the end check NA distribution/number of each column 


KeyboardInterrupt: 

In [ ]:
pre_metadata_df_final

,name,sequence,category,class,source,ref_seq,seq_chr,seq_start,seq_end,seq_strand,variant_class,variant_pos,SPDI,allele,info
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,chr1,2181818,2182138,+,NA,NA,NA,NA,cardiac_neuro_cava_random
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,chr1,2182410,2182738,+,NA,NA,NA,NA,cardiac_neuro_cava_random
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,chr1,2182832,2183099,+,NA,NA,NA,NA,cardiac_neuro_cava_random
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,chr1,2184994,2185331,+,NA,NA,NA,NA,cardiac_neuro_cava_random
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,chr1,2188356,2188693,+,NA,NA,NA,NA,cardiac_neuro_cava_random
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,scrambled,element inactive control,NA,GRCh38,NaN,NaN,NaN,NaN,NA,NA,NA,NA,MK
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,scrambled,element inactive control,NA,GRCh38,NaN,NaN,NaN,NaN,NA,NA,NA,NA,MK
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,scrambled,element inactive control,NA,GRCh38,NaN,NaN,NaN,NaN,NA,NA,NA,NA,MK
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,scrambled,element inactive control,NA,GRCh38,NaN,NaN,NaN,NaN,NA,NA,NA,NA,MK


### Sanity check seq_start, seq_end
- get the length of the regions and compare

In [ ]:
sub_df = pre_metadata_df[pre_metadata_df['tmp_label'] == 'cardiac_neuro_cava_random']
sub_df.loc[:,'tmp_seq_length'] = sub_df['seq_end'].astype(int) - sub_df['seq_start'].astype(int)
sub_df['tmp_seq_length'].value_counts()
sub_df

/tmp/ipykernel_1887/3271522864.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_df.loc[:,'tmp_seq_length'] = sub_df['seq_end'].astype(int) - sub_df['seq_start'].astype(int)


,header,sequence,tmp_label,name,category,class,source,variant_class,variant_pos,SPDI,allele,info,ref_seq,tmp_bed_name,seq_chr,seq_start,seq_end,tmp_score,seq_strand,tmp_seq_length
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",NA,NA,NA,NA,cardiac_neuro_cava_random,GRCh38,SKI|ENSG00000157933.11|EH38E2778476,chr1,2181818,2182138,.,+,320
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",NA,NA,NA,NA,cardiac_neuro_cava_random,GRCh38,SKI|ENSG00000157933.11|EH38E2778477,chr1,2182410,2182738,.,+,328
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",NA,NA,NA,NA,cardiac_neuro_cava_random,GRCh38,SKI|ENSG00000157933.11|EH38E2778478,chr1,2182832,2183099,.,+,267
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",NA,NA,NA,NA,cardiac_neuro_cava_random,GRCh38,SKI|ENSG00000157933.11|EH38E2778480,chr1,2184994,2185331,.,+,337
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",NA,NA,NA,NA,cardiac_neuro_cava_random,GRCh38,SKI|ENSG00000157933.11|EH38E2778484,chr1,2188356,2188693,.,+,337
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73935,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTGGAGCTCTGCCTCACCCCACCTGGCCCCAAT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",SNP,NA,NA,alt,cardiac_neuro_cava_random,GRCh38,G6PD|ENSG00000160211.20|EH38E3949733,chrX,154544972,154545298,.,-,326
73936,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTATGTCTGAATTCACCTCCAAATAATGGGAAA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",SNP,NA,NA,alt,cardiac_neuro_cava_random,GRCh38,G6PD|ENSG00000160211.20|EH38E2774396,chrX,154549862,154550013,.,-,151
73937,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTCCTCTGCCCTCCCTGGCTTCTTCCCCTGTCC...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",SNP,NA,NA,alt,cardiac_neuro_cava_random,GRCh38,G6PD|ENSG00000160211.20|EH38E3949745,chrX,154552118,154552443,.,-,325
73938,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTCCTCTGCCCTCCCTGGCTTCTTCCCCTGTCC...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",SNP,NA,NA,alt,cardiac_neuro_cava_random,GRCh38,G6PD|ENSG00000160211.20|EH38E3949745,chrX,154552118,154552443,.,-,325


#### Investigates similarity of sequences

In [ ]:
AAGAATACAAGTAACTGATGAATGAAGGGGGCATCTTGTGTCCCCACAATCCTGCTGTGCGCACACCACAGGTGAGCCGTTCTGCCTAAGGGAACAGCCCCGGCCCCTCCCTCCGGCTCCTCCCCAGCACCGTCTCCTCCACCCAGTGGCCTGGCCGTGGATGCTGCCTGTGGCCCAGCTTTGAGACACCGCCCTGACACGTGTCCAGCCTTACGTGGAAGGATTTGTCTGTTTTGTGGCATCCTAGTAGATGCCACGTTAGTAGATGCC
AAGAATACAAGTAACTGATGAATGAAGGGGGCATCTTGTGTCCCCACAATCCTGCTGTGCGCACACCACAGGTGAGCCGTTCTGCCTAAGGGAACAGCCCCGGCCCCTCCCTCCGGCTCCTCCCCAGCACCGTCTCCTCCACCCAGTGGCCTGGCCGTGGATGCTGCCTGTGGCCCAGCTTTGAGACACCGCCCTGACACGTGTCCAGCCTTACGTGGAAGGATTTGTCTGTTTTGTGGCATCCTAGTAGATGCCACGTTAGTAGATGCC

In [ ]:
AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGGCATCTTGTGTCCCCACAATCCTGCTGTGCGCACACCACAGGTGAGCCGTTCTGCCTAAGGGAACAGCCCCGGCCCCTCCCTCCGGCTCCTCCCCAGCACCGTCTCCTCCACCCAGTGGCCTGGCCGTGGATGCTGCCTGTGGCCCAGCTTTGAGACACCGCCCTGACACGTGTCCAGCCTTACGTGGAAGGATTTGTCTGTTTTGTGGCATCCTAGTAGATGCCACGTTAGTAGATGCCCATTGCGTGAACCGA

In [ ]:
def get_seq_length(row):
    return len(row['sequence'])

sub_df['tmp_seq_length'] = sub_df.apply(get_seq_length, axis=1)
sub_df['tmp_seq_without_adapter'] = sub_df['sequence'].str.slice(15, 285)

# exclude ALT_ sequences
sub_df = sub_df[~sub_df['header'].str.contains('ALT_')]
sub_df.shape # 27482

# revert sequences on - strand

# # write fasta of tmp_seq_without_adapter and name as header 
# file_path = 'tested_ref_and_elements.fasta'

# for index, row in sub_df.iterrows():
#     with open(file_path, 'a') as f:
#         f.write('>' + row['name'] + '\n' + row['tmp_seq_without_adapter'] + '\n')





# sub_df['tmp_seq_length'].value_counts()

/tmp/ipykernel_1887/3550457708.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_df['tmp_seq_length'] = sub_df.apply(get_seq_length, axis=1)
/tmp/ipykernel_1887/3550457708.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_df['tmp_seq_without_adapter'] = sub_df['sequence'].str.slice(15, 285)


(27482, 21)

In [ ]:
# check strand information of the sequences: 
sub_df['seq_strand'].value_counts() # all are "+"

seq_strand
+    14306
-    13176
Name: count, dtype: int64

#### Investigate blat results: 
<!-- - expect 27482 hits with 100% identity and 270 matching bases -->
- for 27120 we have exactly one match with 270 matching bases
- interest in T_name (=> chromosome), T_start and T_end, strand

In [ ]:
import pandas as pd

In [ ]:
# bash commands to generate a tsv out of the psl file
file = '/data/cephfs-2/unmirrored/groups/ag-kircher/MPRA/kilian_projects/hg38/ncbi_dataset/data/GCF_000001405.26/blat_result_without_header.tsv'
# remove header
# tail -n +6 blat_matched_file.psl > blat_result_without_header.tsv

blat_result_df = pd.read_csv(file, sep='\t', header=None)

# add column names
blat_result_columns = ['match', 'mismatch', 'rep_match', 'Ns', 'Q_gap_count', 'Q_gap_bases', 'T_gap_count', 'T_gap_bases', 'strand', 'Q_name', 'Q_size', 'Q_start', 'Q_end', 'T_name', 'T_size', 'T_start', 'T_end', 'block_count', 'blockSizes', 'qStarts', 'tStarts']

blat_result_df.columns = blat_result_columns

In [ ]:
blat_result_df
# filter: number of matches=270, 
perfect_matches = blat_result_df[blat_result_df['match'] == 270]
perfect_matches.shape # 28276
# filter for blockSizes = 270
perfect_matches = perfect_matches[perfect_matches['blockSizes'] == '270,']
perfect_matches.shape # 28236
# check all columns for unique values and did not find any suspecious values
perfect_matches.T_name.value_counts() # 0 
# filter for matches startwith "NC_" in T_name
perfect_matches = perfect_matches[perfect_matches['T_name'].str.startswith('NC_')]
perfect_matches.shape # 27482
perfect_matches.Q_name.nunique() # 27482

# get subset of interesting columns
interesting_blat_results = ['Q_name', 'match', 'strand', 'T_name', 'T_start', 'T_end']
interesting_blat_results

perfect_matches = perfect_matches[interesting_blat_results]
perfect_matches

,Q_name,match,strand,T_name,T_start,T_end
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,270,+,NC_000001.11,2181843,2182113
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,270,+,NC_000001.11,2182439,2182709
64,cardiac_neuro_cava_random:SKI|ENSG00000157933....,270,+,NC_000001.11,2182830,2183100
122,cardiac_neuro_cava_random:SKI|ENSG00000157933....,270,+,NC_000001.11,2185027,2185297
124,cardiac_neuro_cava_random:SKI|ENSG00000157933....,270,+,NC_000001.11,2188389,2188659
...,...,...,...,...,...,...
369505,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,270,-,NC_000023.11,154531979,154532249
369506,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,270,-,NC_000023.11,154539055,154539325
369507,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,270,-,NC_000023.11,154545000,154545270
369562,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,270,-,NC_000023.11,154549802,154550072


In [ ]:
perfect_matches

,Q_name,match,strand,T_name,T_start,T_end
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,270,+,NC_000001.11,2181843,2182113
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,270,+,NC_000001.11,2182439,2182709
64,cardiac_neuro_cava_random:SKI|ENSG00000157933....,270,+,NC_000001.11,2182830,2183100
122,cardiac_neuro_cava_random:SKI|ENSG00000157933....,270,+,NC_000001.11,2185027,2185297
124,cardiac_neuro_cava_random:SKI|ENSG00000157933....,270,+,NC_000001.11,2188389,2188659
...,...,...,...,...,...,...
369505,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,270,-,NC_000023.11,154531979,154532249
369506,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,270,-,NC_000023.11,154539055,154539325
369507,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,270,-,NC_000023.11,154545000,154545270
369562,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,270,-,NC_000023.11,154549802,154550072


In [ ]:
perfect_matches.T_name.value_counts()

T_name
NC_000001.11    3119
NC_000003.12    1899
NC_000007.14    1816
NC_000002.12    1613
NC_000011.10    1504
NC_000010.11    1448
NC_000016.10    1389
NC_000006.12    1361
NC_000017.11    1331
NC_000009.12    1285
NC_000023.11    1189
NC_000015.10    1162
NC_000012.12    1049
NC_000005.10    1000
NC_000014.9      985
NC_000004.12     923
NC_000018.10     860
NC_000019.10     844
NC_000020.11     804
NC_000008.11     733
NC_000022.11     691
NC_000021.9      286
NC_000013.11     191
Name: count, dtype: int64

##### Add to the table
- check if all + have a + in the new search results
- check if the new positions are close to the old ones
  - same chromosome
  - start - start + end - end 

In [ ]:
def set_modified_chromosome(row):
    """i.e. from NC_000001.11 to chr1, ..."""
    if '23' in row['T_name']:
        return 'chrX'
    if '24' in row['T_name']:
        return 'chrY'
    chr_number = int(row['T_name'].split('_')[1].split('.')[0])
    return 'chr%s'%(chr_number) 

In [ ]:
mapped_blat_results = pre_metadata_df_final.merge(perfect_matches, left_on='name', right_on='Q_name', how='left')

mapped_blat_results.shape[0] - mapped_blat_results['Q_name'].isna().sum()

suc_mapped_blat_results = mapped_blat_results[~mapped_blat_results['Q_name'].isna()]

# check quality of new results: 
# 1. strand same? 
suc_mapped_blat_results['tmp_same_strand_info'] = suc_mapped_blat_results.apply(lambda x: True if x['seq_strand'] == x['strand'] else False, axis=1)
if suc_mapped_blat_results['tmp_same_strand_info'].sum() == suc_mapped_blat_results.shape[0]: # passed the test
    print('Passed test: Strand information is the same')

# 2. Same chromosome?
# convert NC_* chromosomes to chr*
suc_mapped_blat_results['tmp_modified_chr'] = suc_mapped_blat_results.apply(set_modified_chromosome, axis=1)
suc_mapped_blat_results['tmp_same_chr_info'] = suc_mapped_blat_results.apply(lambda x: True if x['seq_chr'] == x['tmp_modified_chr'] else False, axis=1)
if suc_mapped_blat_results['tmp_same_chr_info'].sum() == suc_mapped_blat_results.shape[0]: # passed the test
    print('All results have the same chromosome information')

# Check if positions are nearby 
suc_mapped_blat_results['T_start'] = suc_mapped_blat_results['T_start'].astype(int)
suc_mapped_blat_results['T_end'] = suc_mapped_blat_results['T_end'].astype(int)
suc_mapped_blat_results['seq_start'] = suc_mapped_blat_results['seq_start'].astype(int)
suc_mapped_blat_results['seq_end'] = suc_mapped_blat_results['seq_end'].astype(int)
suc_mapped_blat_results['tmp_start_difference'] = suc_mapped_blat_results.apply(lambda x: abs(x['T_start'] - x['seq_start']), axis=1)
suc_mapped_blat_results['tmp_start_difference'].value_counts()


# print suc_mapped_blat_results value counts as dataframe 
pd.DataFrame(suc_mapped_blat_results['tmp_start_difference'].value_counts())



# suc_mapped_blat_results['tmp_end_difference'] = suc_mapped_blat_results.apply(lambda x: abs(x['T_end'] - x['seq_end']), axis=1)
# start positions have difference from over 1kb this is sus 
# 39    1955
# 38    1508
# 35    1380
# 40    1364
# 37    1231

/tmp/ipykernel_1887/2245177199.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  suc_mapped_blat_results['tmp_same_strand_info'] = suc_mapped_blat_results.apply(lambda x: True if x['seq_strand'] == x['strand'] else False, axis=1)
/tmp/ipykernel_1887/2245177199.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  suc_mapped_blat_results['tmp_modified_chr'] = suc_mapped_blat_results.apply(set_modified_chromosome, axis=1)


Passed test: Strand information is the same
All results have the same chromosome information


/tmp/ipykernel_1887/2245177199.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  suc_mapped_blat_results['tmp_same_chr_info'] = suc_mapped_blat_results.apply(lambda x: True if x['seq_chr'] == x['tmp_modified_chr'] else False, axis=1)
/tmp/ipykernel_1887/2245177199.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  suc_mapped_blat_results['T_start'] = suc_mapped_blat_results['T_start'].astype(int)
/tmp/ipykernel_1887/2245177199.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a s

,count
tmp_start_difference,
39,1955
38,1508
35,1380
40,1364
37,1231
...,...
47,152
48,152
59,150


In [81]:
# write seq_chr, seq_start, seq_end, name, ., seq_strand
suc_mapped_blat_results['score'] = '.'
region_a_table = suc_mapped_blat_results[['seq_chr', 'seq_start', 'seq_end', 'name', 'score', 'seq_strand']]
# # write tmp_modified_chr, T_start, T_end, Q_name, ., strand
region_b_table = suc_mapped_blat_results[['tmp_modified_chr', 'T_start', 'T_end', 'Q_name', 'score', 'strand']]

# logic: check if seq_start < T_start and T_end < seq_end
# if not: check if seq_start < T_start and T_end > seq_end
# print the name and count the number of cases which are printed at the end 
def is_inbetween(row):
    """
    Return True if seq_start < T_start and T_end < seq_end
    Can only be done like that because all are on the same strand (?)
    """
    return row['seq_start'] < row['T_start'] and row['T_end'] < row['seq_end']

suc_mapped_blat_results['tmp_is_inbetween'] = suc_mapped_blat_results.apply(is_inbetween, axis=1)
pd.DataFrame(suc_mapped_blat_results['tmp_is_inbetween'].value_counts())
# 11538 are not inbetween 



/tmp/ipykernel_1887/3931396714.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  suc_mapped_blat_results['score'] = '.'
/tmp/ipykernel_1887/3931396714.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  suc_mapped_blat_results['tmp_is_inbetween'] = suc_mapped_blat_results.apply(is_inbetween, axis=1)


,count
tmp_is_inbetween,
True,15944
False,11538


##### Investigate the ucsc genome browser of these regions and find the cCREs


In [84]:
suc_mapped_blat_results['tmp_region5k_length'] = suc_mapped_blat_results['seq_end'] - suc_mapped_blat_results['seq_start']

## not inbetween
suc_mapped_blat_results[suc_mapped_blat_results['tmp_is_inbetween'] == False]['tmp_region5k_length'].max()

# ### only smaller length:
suc_mapped_blat_results[suc_mapped_blat_results['tmp_region5k_length'] <= 271].shape[0] # 11336 
suc_mapped_blat_results[suc_mapped_blat_results['tmp_region5k_length'] <= 271]['name'].to_list()
# example1: cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778609_fwd_tile1-1
# example2: cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779779_fwd_tile1-1
# example3: cardiac_neuro_cava_random:FGF12|ENSG00000114279.15|EH38E2269809_rev_tile1-1
suc_mapped_blat_results[suc_mapped_blat_results['name'] == 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778609_fwd_tile1-1']
# # on the last 15bp on the right a distal enhancer like sequence could be found: 
# # investigated with ucsc genome browser: https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr1%3A2250503%2D2250773&hgsid=2082689898_VAczbUehrymUlkaWiiYiCorrQZsL
suc_mapped_blat_results[suc_mapped_blat_results['name'] == 'cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779779_fwd_tile1-1']
# # Didn't find regulatory element at this position
# # https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr1%3A3210194%2D3210464&hgsid=2082689898_VAczbUehrymUlkaWiiYiCorrQZsL 

# suc_mapped_blat_results[suc_mapped_blat_results['name'] == 'cardiac_neuro_cava_random:FGF12|ENSG00000114279.15|EH38E2269809_rev_tile1-1']
# suc_mapped_blat_results[suc_mapped_blat_results['name'] == 'cardiac_neuro_cava_random:FGF12|ENSG00000114279.15|EH38E2269809_rev_tile1-1']['sequence'].values[0][15:]
# - strand hit: go to right side end and use complement bases: blat coordinates: NC_000003.12	192748482	192748752
# # found ELS from encode (matching accessions: EH38E2269809): chr3:192,748,528-192,748,708 => we are out of bounds at the right side
# # https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr3%3A192748528%2D192748752&hgsid=2082689898_VAczbUehrymUlkaWiiYiCorrQZsL

# suc_mapped_blat_results[suc_mapped_blat_results['name'] == 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778717_fwd_tile1-1']
# suc_mapped_blat_results[suc_mapped_blat_results['name'] == 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778717_fwd_tile1-1']['sequence'].values[0][15:]
# # + strand hit: found cCRE from encode accession:EH38E1311783: chr1:2309913-2310122 our position (region5K): chr1	2309925	2310126 blat: NC_000001.11	2309890	2310160
# # not matching accession: our: EH38E2778717 found: EH38E1311783
# # https://genome.ucsc.edu/cgi-bin/hgc?hgsid=2080448280_bDXQ4noYZFZPU5Nj0a4cUrr9C1wq&db=hg38&c=chr1&l=2309924&r=2310126&o=2309912&t=2310122&g=encodeCcreCombined&i=EH38E1311783

# # example chr X: cardiac_neuro_cava_random:REF_G6PD|ENSG00000160211.20|EH38E3949700_rev_tile1-1
# suc_mapped_blat_results[suc_mapped_blat_results['name'] == 'cardiac_neuro_cava_random:REF_G6PD|ENSG00000160211.20|EH38E3949700_rev_tile1-1']
# suc_mapped_blat_results[suc_mapped_blat_results['name'] == 'cardiac_neuro_cava_random:REF_G6PD|ENSG00000160211.20|EH38E3949700_rev_tile1-1']['sequence'].values[0][15:]
# # found CRE from encode but it is larger than the region and encode accession does not match (chrX:154515117-154515465; EH38E2774357 (https://genome.ucsc.edu/cgi-bin/hgc?hgsid=2080448280_bDXQ4noYZFZPU5Nj0a4cUrr9C1wq&db=hg38&c=chrX&l=154515132&r=154515183&o=154515116&t=154515465&g=encodeCcreCombined&i=EH38E2774357))
# # region5K: https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chrX%3A154515118%2D154515418&hgsid=2080448280_bDXQ4noYZFZPU5Nj0a4cUrr9C1wq
# # blat region of sequence (NC_000023.11	154515133	154515403): https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chrX%3A154515133%2D154515403&hgsid=2080448280_bDXQ4noYZFZPU5Nj0a4cUrr9C1wq


/tmp/ipykernel_1887/655454006.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  suc_mapped_blat_results['tmp_region5k_length'] = suc_mapped_blat_results['seq_end'] - suc_mapped_blat_results['seq_start']


,name,sequence,category,class,source,ref_seq,seq_chr,seq_start,seq_end,seq_strand,...,T_name,T_start,T_end,tmp_same_strand_info,tmp_modified_chr,tmp_same_chr_info,tmp_start_difference,score,tmp_is_inbetween,tmp_region5k_length
52,cardiac_neuro_cava_random:PRDM16|ENSG000001426...,AGGACCGGATCAACTAGCCAGCCTGCACTCACAGGCAGGTGGGCTC...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,chr1,3210229,3210430,+,...,NC_000001.11,3210194,3210464,True,chr1,True,35,.,False,201


In [62]:
suc_mapped_blat_results[suc_mapped_blat_results['seq_start'] == 154515118]['name'].to_list()
# suc_mapped_blat_results[suc_mapped_blat_results['seq_start'] == 154515118]['name'].to_list()

['cardiac_neuro_cava_random:REF_G6PD|ENSG00000160211.20|EH38E3949700_rev_tile1-1']

In [46]:
suc_mapped_blat_results[suc_mapped_blat_results['tmp_start_difference'] == 39].iloc[23,:]
## sanity check: 
# - check inbetween
# suc_mapped_blat
# - check for 2 regions

name                    cardiac_neuro_cava_random:RERE|ENSG00000142599...
sequence                AGGACCGGATCAACTGATTAAGCTCCTATATGATATATGTAACAAA...
category                                                          element
class                                                                test
source                  candidate CRE nearby cardiac, neuro, cava and ...
ref_seq                                                            GRCh38
seq_chr                                                              chr1
seq_start                                                         8733505
seq_end                                                           8733853
seq_strand                                                              -
variant_class                                                          NA
variant_pos                                                            NA
SPDI                                                                   NA
allele                                

In [69]:
perfect_matches

,match,mismatch,rep_match,Ns,Q_gap_count,Q_gap_bases,T_gap_count,T_gap_bases,strand,Q_name,...,Q_start,Q_end,T_name,T_size,T_start,T_end,block_count,blockSizes,qStarts,tStarts
0,270,0,0,0,0,0,0,0,+,cardiac_neuro_cava_random:SKI|ENSG00000157933....,...,0,270,NC_000001.11,248956422,2181843,2182113,1,"270,","0,","2181843,"
1,270,0,0,0,0,0,0,0,+,cardiac_neuro_cava_random:SKI|ENSG00000157933....,...,0,270,NC_000001.11,248956422,2182439,2182709,1,"270,","0,","2182439,"
64,270,0,0,0,0,0,0,0,+,cardiac_neuro_cava_random:SKI|ENSG00000157933....,...,0,270,NC_000001.11,248956422,2182830,2183100,1,"270,","0,","2182830,"
122,270,0,0,0,0,0,0,0,+,cardiac_neuro_cava_random:SKI|ENSG00000157933....,...,0,270,NC_000001.11,248956422,2185027,2185297,1,"270,","0,","2185027,"
124,270,0,0,0,0,0,0,0,+,cardiac_neuro_cava_random:SKI|ENSG00000157933....,...,0,270,NC_000001.11,248956422,2188389,2188659,1,"270,","0,","2188389,"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
369505,270,0,0,0,0,0,0,0,-,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,...,0,270,NC_000023.11,156040895,154531979,154532249,1,"270,","0,","154531979,"
369506,270,0,0,0,0,0,0,0,-,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,...,0,270,NC_000023.11,156040895,154539055,154539325,1,"270,","0,","154539055,"
369507,270,0,0,0,0,0,0,0,-,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,...,0,270,NC_000023.11,156040895,154545000,154545270,1,"270,","0,","154545000,"
369562,270,0,0,0,0,0,0,0,-,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,...,0,270,NC_000023.11,156040895,154549802,154550072,1,"270,","0,","154549802,"


In [47]:
perfect_matches.groupby('Q_name').size().value_counts()
# group by Q_name
# 1     27120
# 2       221
# 3        83
# 10       32
# 4        17
# 9         8
# 5         1
perfect_matches.groupby(['Q_name', 'strand']).size().value_counts()
# groupby 'Q_name', 'strand'
# 1     27147
# 2       246
# 3        67
# 10       32
# 4        11
# 9         8

# which sequences have multiple matches
multiple_matches = perfect_matches.groupby('Q_name').size()
multiple_matches = multiple_matches[multiple_matches > 1]

In [48]:
multiple_matches

Q_name
cardiac_neuro_cava_random:AP3B2|ENSG00000103723.17|EH38E3150740_rev_tile1-1     2
cardiac_neuro_cava_random:AP3B2|ENSG00000103723.17|EH38E3150741_rev_tile1-1     2
cardiac_neuro_cava_random:ASH1L|ENSG00000116539.14|EH38E2839971_rev_tile1-1     2
cardiac_neuro_cava_random:CNOT3|ENSG00000088038.20|EH38E1963947_fwd_tile1-1    10
cardiac_neuro_cava_random:CNOT3|ENSG00000088038.20|EH38E3316435_fwd_tile1-1    10
                                                                               ..
cardiac_neuro_cava_random:TCF20|ENSG00000100207.21|EH38E3484239_rev_tile1-1     3
cardiac_neuro_cava_random:TCF20|ENSG00000100207.21|EH38E3484241_rev_tile1-1     3
cardiac_neuro_cava_random:TCF20|ENSG00000100207.21|EH38E3484245_rev_tile1-1     3
cardiac_neuro_cava_random:TCF20|ENSG00000100207.21|EH38E3484247_rev_tile1-1     3
cardiac_neuro_cava_random:TCF20|ENSG00000100207.21|EH38E3484250_rev_tile1-1     3
Length: 362, dtype: int64

In [104]:
sub_df['tmp_seq_without_adapter'].to_list()

['...',
 ...]

In [ ]:
# download hg38 reference genome

In [107]:
! curl -L 'https://api.genome.ucsc.edu/search?search=AAGAATACAAGTAACTGATGAATGAAGGGGGCATCTTGTGTCCCCACAATCCTGCTGTGCGCACACCACAGGTGAGCCGTTCTGCCTAAGGGAACAGCCCCGGCCCCTCCCTCCGGCTCCTCCCCAGCACCGTCTCCTCCACCCAGTGGCCTGGCCGTGGATGCTGCCTGTGGCCCAGCTTTGAGACACCGCCCTGACACGTGTCCAGCCTTACGTGGAAGGATTTGTCTGTTTTGTGGCATCCTAGTAGATGCCACGTTAGTAGATGCC&genome=hg38'


{ "downloadTime": "2024:03:27T19:22:03Z", "downloadTimeStamp": 1711567323, "genome": "hg38"} 

#### Underscore counts in the headers: We want to combine these headers with the header in the bed file [here](/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/final_design/input/regions_5K.bed)
n_underscore
7    46458
6    18582

In [30]:
variant_list_test_seqs = pre_metadata_df_final[pre_metadata_df_final['category'] == 'variant']['name'].to_list() # e.g. 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1'

def get_region_match_name(header):
    """Get the region match name from the header
    1. split by ':' and take the second part [1]
    2. split by '_' and take the second part [1]
    """
    if header.split(':')[0] == 'cardiac_neuro_cava_random':
        if 'ALT_' in header or 'REF_' in header:
            if '~' in header:
                # replace ~ to ,: cardiac_neuro_cava_random:REF_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1
                header = header.replace('~', ',')
            # easy types: 
            # cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1'
            # cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1'
            return ':'.join(header.split(':')[1:]).split('_')[1]
        else: # cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778480_fwd_tile1-1
            if '~' in header:
                # replace ~ to ,: cardiac_neuro_cava_random:APOL2|ENSG00000128335.14|EH38E3478577~APOL4|ENSG00000100336.18|EH38E3478577_rev_tile1-1
                header = header.replace('~', ',')
            return ':'.join(header.split(':')[1:]).split('_')[0]
    else:
        return header

def get_variant_position(header):
    """Check the vcf file, prepare ID and merge with the tested sequences"""
    # /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/final_design/input/variants_5K.vcf
    # G6PD|ENSG00000160211.20|EH38E3949659|X-154478369-G-A

# investigate different groups of headers depending on the nuumber of "_" used. So count the number of "_" in name
tested_variants = pre_metadata_df_final[pre_metadata_df_final['name'].str.split(':').str[0] == 'cardiac_neuro_cava_random']
# tested_variants = tested_variants[tested_variants['category'] == 'variant']
tested_variants['n_underscore'] = tested_variants['name'].str.count('_')
tested_variants['n_underscore'].value_counts()

# # show one case for each number of underscores
# for i in [6,7]:
#     print(i)
#     print(tested_variants[tested_variants['n_underscore'] == i]['name'].to_list())



# ## now make from all cardiac_neuro_cava_random headers the following mutations and call it bed_match_name
tested_variants['region_match_name'] = tested_variants['name'].apply(get_region_match_name)
tested_variants[tested_variants['name'] == 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778476_fwd_tile1-1']

# 3. test how many sequences can be matched with the bed file
bed_file_test_seqs = pd.read_csv(config['files']['final_design']['test_seqs_region_file'], sep='\t', header=None)
bed_file_test_seqs.columns = ['tmp_chrom', 'tmp_chromStart', 'tmp_chromEnd', 'tmp_name', 'tmp_score', 'tmp_strand']
bed_file_test_seqs

# left join tested_variants and bed_file both files and count number of NAs (not found in bed file)
checking_chromposrefalt = tested_variants.merge(bed_file_test_seqs, left_on='region_match_name', right_on='tmp_name', how='left')
checking_chromposrefalt # left: 73940 outer: 86376
# checking_chromposrefalt['tmp_chrom'].isna().sum() # 8900 => all elements are not in the bed file?

# # # checking sequences with NA values (alle haben so eine ~)
# checking_chromposrefalt[checking_chromposrefalt['tmp_chrom'].isna()]['name'].to_list()

# replaced ~ with , and now no NA in tested sequences matching with the bed file
# next step: add this step earlier to the generation script

/tmp/ipykernel_45845/2932931991.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tested_variants['n_underscore'] = tested_variants['name'].str.count('_')
/tmp/ipykernel_45845/2932931991.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tested_variants['region_match_name'] = tested_variants['name'].apply(get_region_match_name)


,name,sequence,category,class,source,ref_sequence,chrom,chrom_start,chrom_end,variant_class,...,allele,info,n_underscore,region_match_name,tmp_chrom,tmp_chromStart,tmp_chromEnd,tmp_name,tmp_score,tmp_strand
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,NA,...,NA,cardiac_neuro_cava_random,5,SKI|ENSG00000157933.11|EH38E2778476,chr1,2181818,2182138,SKI|ENSG00000157933.11|EH38E2778476,.,+
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,NA,...,NA,cardiac_neuro_cava_random,5,SKI|ENSG00000157933.11|EH38E2778477,chr1,2182410,2182738,SKI|ENSG00000157933.11|EH38E2778477,.,+
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,NA,...,NA,cardiac_neuro_cava_random,5,SKI|ENSG00000157933.11|EH38E2778478,chr1,2182832,2183099,SKI|ENSG00000157933.11|EH38E2778478,.,+
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,NA,...,NA,cardiac_neuro_cava_random,5,SKI|ENSG00000157933.11|EH38E2778480,chr1,2184994,2185331,SKI|ENSG00000157933.11|EH38E2778480,.,+
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,NA,...,NA,cardiac_neuro_cava_random,5,SKI|ENSG00000157933.11|EH38E2778484,chr1,2188356,2188693,SKI|ENSG00000157933.11|EH38E2778484,.,+
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73935,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTGGAGCTCTGCCTCACCCCACCTGGCCCCAAT...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,SNP,...,alt,cardiac_neuro_cava_random,7,G6PD|ENSG00000160211.20|EH38E3949733,chrX,154544972,154545298,G6PD|ENSG00000160211.20|EH38E3949733,.,-
73936,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTATGTCTGAATTCACCTCCAAATAATGGGAAA...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,SNP,...,alt,cardiac_neuro_cava_random,7,G6PD|ENSG00000160211.20|EH38E2774396,chrX,154549862,154550013,G6PD|ENSG00000160211.20|EH38E2774396,.,-
73937,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTCCTCTGCCCTCCCTGGCTTCTTCCCCTGTCC...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,SNP,...,alt,cardiac_neuro_cava_random,7,G6PD|ENSG00000160211.20|EH38E3949745,chrX,154552118,154552443,G6PD|ENSG00000160211.20|EH38E3949745,.,-
73938,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTCCTCTGCCCTCCCTGGCTTCTTCCCCTGTCC...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,SNP,...,alt,cardiac_neuro_cava_random,7,G6PD|ENSG00000160211.20|EH38E3949745,chrX,154552118,154552443,G6PD|ENSG00000160211.20|EH38E3949745,.,-


In [19]:
tested_variants[tested_variants['category'] == 'variant']['region_match_name'].to_list()

['SKI|ENSG00000157933.11|EH38E2778471',
 'SKI|ENSG00000157933.11|EH38E2778490',
 'SKI|ENSG00000157933.11|EH38E2778492',
 'SKI|ENSG00000157933.11|EH38E1311587',
 'SKI|ENSG00000157933.11|EH38E2778494',
 'SKI|ENSG00000157933.11|EH38E2778498',
 'SKI|ENSG00000157933.11|EH38E2778506',
 'SKI|ENSG00000157933.11|EH38E2778513',
 'SKI|ENSG00000157933.11|EH38E2778524',
 'SKI|ENSG00000157933.11|EH38E2778530',
 'SKI|ENSG00000157933.11|EH38E2778534',
 'SKI|ENSG00000157933.11|EH38E2778535',
 'SKI|ENSG00000157933.11|EH38E2778544',
 'SKI|ENSG00000157933.11|EH38E2778546',
 'SKI|ENSG00000157933.11|EH38E2778564',
 'SKI|ENSG00000157933.11|EH38E2778568',
 'SKI|ENSG00000157933.11|EH38E2778574',
 'SKI|ENSG00000157933.11|EH38E2778579',
 'SKI|ENSG00000157933.11|EH38E2778583',
 'SKI|ENSG00000157933.11|EH38E2778585',
 'SKI|ENSG00000157933.11|EH38E2778586',
 'SKI|ENSG00000157933.11|EH38E2778588',
 'SKI|ENSG00000157933.11|EH38E2778590',
 'SKI|ENSG00000157933.11|EH38E2778592',
 'SKI|ENSG00000157933.11|EH38E1311678',


In [11]:
variant_list_test_seqs = pre_metadata_df_final[pre_metadata_df_final['category'] == 'variant']['name'].to_list() # e.g. 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1'

def get_region_match_name(header):
    """Get the region match name from the header
    1. split by ':' and take the second part [1]
    2. split by '_' and take the second part [1]
    """
    if header.str.('cardiac_neuro_cava_random'):
        return header.split(':')[1].split('_')[1]
    else:
        return header

# investigate different groups of headers depending on the nuumber of "_" used. So count the number of "_" in name
tested_variants = pre_metadata_df_final[pre_metadata_df_final['name'].str.split(':').str[0] == 'cardiac_neuro_cava_random']
tested_variants = tested_variants[tested_variants['category'] == 'variant']
tested_variants['n_underscore'] = tested_variants['name'].str.count('_')
tested_variants['n_underscore'].value_counts()

# show one case for each number of underscores
for i in [6,7]:
    print(i)
    print(tested_variants[tested_variants['n_underscore'] == i].head(2))



# ## now make from all cardiac_neuro_cava_random headers the following mutations and call it bed_match_name
# # 1. split by ':' and take the second part [1]
# # 2. split by '_' and take the second part [1]
# pre_metadata_df_final['name'].apply(get_region_match_name)

# 3. test how many sequences can be matched with the bed file

SyntaxError: invalid syntax (2013555655.py, line 8)

In [ ]:
variant_list_test_seqs

In [7]:
positive_controls = pre_metadata_df_final[pre_metadata_df_final['info'].isin(positive_control_groups)]
positive_controls


,name,sequence,category,class,source,ref_sequence,chrom,chrom_start,chrom_end,variant_class,variant_pos,SPDI,allele,info
76330,C_positive_neuron_CD:p1_rs6813360_A_C_ref_50_A...,AGGACCGGATCAACTGATTGTATAAAGAAAATGTGTATACACACAC...,variant,variant positive control,NA,GRCh38,NA,NA,NA,SNP,NA,NA,NA,C_positive_neuron_CD
76331,C_positive_neuron_CD:p1_rs55985730_T_G_alt_50_...,AGGACCGGATCAACTGGCGCCTGTAGTCCCAGCTACTTGGGAGGCT...,variant,variant positive control,NA,GRCh38,NA,NA,NA,SNP,NA,NA,NA,C_positive_neuron_CD
76332,C_positive_neuron_CD:c1_NA_NA_NA_NA::72hr_top_...,AGGACCGGATCAACTCTCCCCCGCACAGCGCAGGCTCTCACTGGGA...,element,element active control,NA,GRCh38,NA,NA,NA,NA,NA,NA,NA,C_positive_neuron_CD
76333,C_positive_neuron_CD:p1_rs7115714_G_A_ref_50_A...,AGGACCGGATCAACTTGGTAATTAAAAGCAAAGAGATCTTTTCTAT...,variant,variant positive control,NA,GRCh38,NA,NA,NA,SNP,NA,NA,NA,C_positive_neuron_CD
76334,C_positive_neuron_CD:p1_rs9975055_T_G_alt_50_G...,AGGACCGGATCAACTCCCTGCTCCCCAGTTCCCACCAGAAACCCCA...,variant,variant positive control,NA,GRCh38,NA,NA,NA,SNP,NA,NA,NA,C_positive_neuron_CD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76614,C_positive_neuron_NP:GW18_PFC_ABC_chr12_121536...,AGGACCGGATCAACTGCAGCGCGCCGGCACGAGTGGAGATAATGCG...,element,element active control,NA,GRCh38,NA,NA,NA,NA,NA,NA,NA,C_positive_neuron_NP
76615,C_positive_neuron_NP:NGN2_iPSC_ABC_chr4_850290...,AGGACCGGATCAACTTAACAGATGAAGCCTCAATTAGGTAGCTAGC...,element,element active control,NA,GRCh38,NA,NA,NA,NA,NA,NA,NA,C_positive_neuron_NP
76616,C_positive_neuron_NP:GW18_PFC_ABC_NGN2_iPSC_AB...,AGGACCGGATCAACTCCGAGACCCGCACTTAGTTACCTGAGAAGGC...,element,element active control,NA,GRCh38,NA,NA,NA,NA,NA,NA,NA,C_positive_neuron_NP
76617,C_positive_neuron_NP:Fetal_Cerebrum_Cicero_GW1...,AGGACCGGATCAACTCAGCTCCAGCGGCATGAAATATTGATGCCCT...,element,element active control,NA,GRCh38,NA,NA,NA,NA,NA,NA,NA,C_positive_neuron_NP


In [6]:
def is_control(row):
    """True if control false if not"""
    return row['class'] != 'test'

def get_control_variant(row):
    """Get the control variant"""
    if row['class'] == 'variant positive control':
        return True
    elif row['class'] == 'variant negative control':
        return True
    else:
        return False
    
def is_dnase_control(row):
    """True if control is a dnase control"""
    return 'dnase' in row['info'].lower()

In [7]:
# get number of controls with variants
control_vars = pre_metadata_df_final[pre_metadata_df_final.apply(is_control, axis=1)] # 1058 
# control_vars['info'].nunique() # 8 controls with variants # but CD is wrong
control_vars

,name,sequence,category,class,source,ref_sequence,chrom,chrom_start,chrom_end,variant_class,variant_pos,SPDI,allele,info
73940,GC_Atrial_fib:rs7795510|CAV1|STARR-seq-AF~rs78...,AGGACCGGATCAACTTCATTTCATTATAATCAAAAAGGATTTTTAA...,element,element inactive control,IGVF general controls (GC_Atrial_fib),GRCh38,NA,NA,NA,NA,NA,NA,NA,GC_Atrial_fib
73941,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,AGGACCGGATCAACTCAGCTGCCCATGCTGGGACTGTGATTTTTTG...,variant,variant negative control,IGVF general controls (GC_Atrial_fib),GRCh38,NA,NA,NA,SNP,NA,NA,ref,GC_Atrial_fib
73942,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,AGGACCGGATCAACTAGAGCCCTTCTGGGGGCCCTGGCCACTGGCC...,variant,variant negative control,IGVF general controls (GC_Atrial_fib),GRCh38,NA,NA,NA,SNP,NA,NA,ref,GC_Atrial_fib
73943,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,AGGACCGGATCAACTAGGAAAGGCACTGGAAATTGTACTTACTCCA...,variant,variant negative control,IGVF general controls (GC_Atrial_fib),GRCh38,NA,NA,NA,SNP,NA,NA,ref,GC_Atrial_fib
73944,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,AGGACCGGATCAACTTTTGCAAAGGTATGGTTGGTGGATGGAGAAA...,variant,variant negative control,IGVF general controls (GC_Atrial_fib),GRCh38,NA,NA,NA,SNP,NA,NA,ref,GC_Atrial_fib
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,scrambled,element inactive control,NA,GRCh38,NA,NA,NA,NA,NA,NA,NA,MK
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,scrambled,element inactive control,NA,GRCh38,NA,NA,NA,NA,NA,NA,NA,MK
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,scrambled,element inactive control,NA,GRCh38,NA,NA,NA,NA,NA,NA,NA,MK
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,scrambled,element inactive control,NA,GRCh38,NA,NA,NA,NA,NA,NA,NA,MK


In [8]:
control_vars['info'].unique()

array(['GC_Atrial_fib', 'GC_Liang', 'GC_Selvarajan', 'GC_Mohlke',
       'GC_Kircher', 'GC_Mendelian_variants', 'C_positive_heart_CAD',
       'GC_Cort_Chengyu', 'GC_GABA_Chengyu', 'GC_Glut_Chengyu', 'GC_Hon',
       'GC_Vista',
       'GC_DNase_positive;wrong genome build (GRCh37) used for these sequences',
       'GC_DNase_negative_brain;wrong genome build (GRCh37) used for these sequences',
       'GC_DNase_negative_blood;wrong genome build (GRCh37) used for these sequences',
       'C_negative_heart_MK', 'C_negative_neuron_MK',
       'C_negative_neuron_NP', 'C_positive_heart_MK',
       'C_positive_neuron_CD', 'C_positive_neuron_MK',
       'C_positive_neuron_NP', 'C_positive_heart_AB', 'C_SLEA',
       'GC_DNase_positive_shuffeled;wrong genome build (GRCh37) used for these sequences',
       'GC_DNase_negative_brain_shuffeled;wrong genome build (GRCh37) used for these sequences',
       'GC_DNase_negative_blood_shuffeled;wrong genome build (GRCh37) used for these sequences',
    

In [ ]:
GC_Atrial_fib', 'GC_Liang', 'GC_Selvarajan', 'GC_Mohlke',
       'GC_Kircher', 
       'GC_Cort_Chengyu', 
       'GC_GABA_Chengyu', 'GC_Glut_Chengyu', 'GC_Hon',
       'GC_Mendelian_variants', 'C_positive_heart_CAD',
       'GC_Vista',
       'GC_DNase_positive;wrong genome build (GRCh37) used for these sequences',
       'GC_DNase_negative_brain;wrong genome build (GRCh37) used for these sequences',
       'GC_DNase_negative_blood;wrong genome build (GRCh37) used for these sequences',
       'MK'
       
       'C_negative_heart_MK', 'C_negative_neuron_MK',
       'C_negative_neuron_NP', 'C_positive_heart_MK',
       'C_positive_neuron_CD', 'C_positive_neuron_MK',
       'C_positive_neuron_NP', 'C_positive_heart_AB', 'C_SLEA',
       'GC_DNase_positive_shuffeled;wrong genome build (GRCh37) used for these sequences',
       'GC_DNase_negative_brain_shuffeled;wrong genome build (GRCh37) used for these sequences',
       'GC_DNase_negative_blood_shuffeled;wrong genome build (GRCh37) used for these sequences',


In [20]:
pre_metadata_df_final.apply(is_dnase_control, axis=1).sum()

161

In [16]:
6275 - 289

5986

In [7]:
pre_metadata_df_final.columns

Index(['name', 'sequence', 'category', 'class', 'source', 'ref_sequence',
       'chrom', 'chrom_start', 'chrom_end', 'variant_class', 'variant_pos',
       'SPDI', 'allele', 'info'],
      dtype='object')

### Make REF ALT map (variant map) for controls
- Find all REF and ALT headers and match them for each group (because of not matching patterns)
- the following control groups have variants (positive or negative) (ref_ or alt_ in header.lower())
  - C_positive_heart_CAD
  - C_positive_neuron_CD
  - GC_Atrial_fib
  - GC_Kircher
  - GC_Liang
  - GC_Mendelian_variants
  - GC_Mohlke
  - GC_Selvarajan

In [5]:
old_metadata = config['files']['creating']['old_metadata_table']
pre_metadata_df_final = pd.read_csv(old_metadata, sep="\t")


In [6]:
variant_control_groups = ['C_positive_heart_CAD', 'C_positive_neuron_CD', 'GC_Atrial_fib', 'GC_Kircher', 'GC_Liang', 'GC_Mendelian_variants', 'GC_Mohlke', 'GC_Selvarajan']
# get all entries of pre_metadata_df_final with the labels in the info column with variant in the name
variant_control_seqs = pre_metadata_df_final[pre_metadata_df_final['info'].isin(variant_control_groups)]
variant_control_vars = variant_control_seqs[variant_control_seqs['category'] == 'variant']
variant_control_vars # 1058 

variant_control_vars['class'].value_counts()

class
variant negative control    967
variant positive control     91
Name: count, dtype: int64

In [7]:
variant_control_vars['info'].value_counts()

info
GC_Selvarajan            364
GC_Mendelian_variants    209
GC_Kircher               203
C_positive_heart_CAD      97
C_positive_neuron_CD      91
GC_Atrial_fib             44
GC_Mohlke                 34
GC_Liang                  16
Name: count, dtype: int64

In [ ]:
# info                   class                     number of variants   after assignment (with elements)
# C_positive_heart_CAD   variant negative control     97    92
# C_positive_neuron_CD   variant positive control     91    88
# GC_Atrial_fib          variant negative control     44    45
# GC_Kircher             variant negative control    203    185
# GC_Liang               variant negative control     16    16
# GC_Mendelian_variants  variant negative control    209    185
# GC_Mohlke              variant negative control     34    31
# GC_Selvarajan          variant negative control    364    356

#### C_positive_heart_CAD (97)
- REF: C_positive_heart_CAD:REF_rs17114036 
- ALT: C_positive_heart_CAD:ALT_rs17114036_rs17114036	

In [10]:
def get_matching_element_between_ref_alt(header):
    """Get the rsid from the header"""
    if 'C_positive_heart_CAD' in header:
        return header.split('_')[-1] # C_positive_heart_CAD:ALT_rs17114036_rs17114036
    elif 'GC_Selvarajan' in header:
        # think about tiling: add the tile number to the rsid with an _tile<number>
        tile_number = header.split('tile')[-1].split('-')[0]
        return '%s_tile%s'%(header.split('_')[2].split('|')[0], tile_number) # GC_Selvarajan:REF_rs2993510|STARR-seq-HepG2~rs72856440|STARR-seq-HepG2_fwd_tile1-3
    elif 'GC_Liang' in header:
        return header.split('_')[2].split('|')[0] # GC_Liang:REF_rs2125358|Liang_fwd_tile1-1
    else:
        print('unexpected')

def get_rsID_from_GC_Selvarajan(header, number=0):
    """Get the rsid from the header"""
    if number == 0:
        return get_matching_element_between_ref_alt(header)
    else:
        substr_header = header.split('~')[number]
    return substr_header.split('|')[0]

In [12]:
# example for C_positive_heart_CAD
from collections import defaultdict
 
C_positive_heart_CAD = variant_control_vars[variant_control_vars['info'].isin(['C_positive_heart_CAD'])]
header_list = C_positive_heart_CAD['name'].to_list()


# Go over each REF: put sequence into dict and if rsID is matching (rsID=ALT_<rsID>) then put into dict and compare
# each ref should have at least 1 element in dict
header_list.sort(reverse=True) # first REF then ALT (headers are not different until "ALT_" or "REF_")
ref_dict = defaultdict(list)
for header in header_list:
    if 'REF_' in header: # refs need to be unique
        ref_unique_element = get_matching_element_between_ref_alt(header)
        for alt_header in header_list:
            if 'ALT_' in alt_header:
                alt_unique_element = get_matching_element_between_ref_alt(alt_header)
                if ref_unique_element == alt_unique_element:
                    ref_dict[header].append(alt_header)
    # break
ref_dict

# # investigate number of elements in ref_dict values
# for ref, alt in ref_dict.items():
#     if len(alt) != 1:
#         print(ref)

# prepare dataframe with ref and alt from ref_dict
ref_alt_df = pd.DataFrame(columns=['ID', 'REF', 'ALT'])
id_list = []
ref_list = []
alt_list = []
for ref, alt in ref_dict.items():
    for elem in alt:
        unique_id = get_matching_element_between_ref_alt(ref)
        unique_header = '%s:%s'%(get_label(ref), unique_id)
        id_list.append(unique_header)
        ref_list.append(ref)
        alt_list.append(elem)
ref_alt_df['ID'] = id_list
ref_alt_df['REF'] = ref_list
ref_alt_df['ALT'] = alt_list
ref_alt_df
outpath = 'results/variant_control_map/variant_control_ref_alt_%s.tsv'%('C_positive_heart_CAD')
ref_alt_df.to_csv(outpath, sep='\t', index=False)
    

#### For the easy ones: C_positive_heart_CAD, GC_Liang

In [13]:
from collections import defaultdict

for group in ['C_positive_heart_CAD', 'GC_Liang']:
    group_vars = variant_control_vars[variant_control_vars['info'] == group]
    header_list = group_vars['name'].to_list()


    # Go over each REF: put sequence into dict and if rsID is matching (rsID=ALT_<rsID>) then put into dict and compare
    # each ref should have at least 1 element in dict
    header_list.sort(reverse=True) # first REF then ALT (headers are not different until "ALT_" or "REF_")
    ref_dict = defaultdict(list)
    for header in header_list:
        if 'REF_' in header: # refs need to be unique
            ref_unique_element = get_matching_element_between_ref_alt(header)
            for alt_header in header_list:
                if 'ALT_' in alt_header:
                    alt_unique_element = get_matching_element_between_ref_alt(alt_header)
                    if ref_unique_element == alt_unique_element:
                        ref_dict[header].append(alt_header)


    # # investigate number of elements in ref_dict values
    # for ref, alt in ref_dict.items():
    #     if len(alt) != 1:
    #         print(ref)

    # prepare dataframe with ref and alt from ref_dict
    ref_alt_df = pd.DataFrame(columns=['ID', 'REF', 'ALT'])
    id_list = []
    ref_list = []
    alt_list = []
    for ref, alt in ref_dict.items():
        for elem in alt:
            unique_id = get_matching_element_between_ref_alt(ref)
            unique_header = '%s:%s'%(get_label(ref), unique_id)
            id_list.append(unique_header)
            ref_list.append(ref)
            alt_list.append(elem)
    ref_alt_df['ID'] = id_list
    ref_alt_df['REF'] = ref_list
    ref_alt_df['ALT'] = alt_list
    ref_alt_df
    outpath = 'results/variant_control_map/variant_control_ref_alt_%s.tsv'%(group)
    ref_alt_df.to_csv(outpath, sep='\t', index=False)

In [ ]:
from collections import defaultdict
 
C_positive_heart_CAD = variant_control_vars[variant_control_vars['info'].isin(['C_positive_heart_CAD'])]
header_list = C_positive_heart_CAD['name'].to_list()


# Go over each REF: put sequence into dict and if rsID is matching (rsID=ALT_<rsID>) then put into dict and compare
# each ref should have at least 1 element in dict
header_list.sort(reverse=True) # first REF then ALT (headers are not different until "ALT_" or "REF_")
ref_dict = defaultdict(list)
for header in header_list:
    if 'REF_' in header: # refs need to be unique
        ref_unique_element = get_matching_element_between_ref_alt(header)
        for alt_header in header_list:
            if 'ALT_' in alt_header:
                alt_unique_element = get_matching_element_between_ref_alt(alt_header)
                if ref_unique_element == alt_unique_element:
                    ref_dict[header].append(alt_header)
    # break
ref_dict

# # investigate number of elements in ref_dict values
# for ref, alt in ref_dict.items():
#     if len(alt) != 1:
#         print(ref)

# prepare dataframe with ref and alt from ref_dict
ref_alt_df = pd.DataFrame(columns=['ID', 'REF', 'ALT'])
id_list = []
ref_list = []
alt_list = []
for ref, alt in ref_dict.items():
    for elem in alt:
        unique_id = get_matching_element_between_ref_alt(ref)
        unique_header = '%s:%s'%(get_label(ref), unique_id)
        id_list.append(unique_header)
        ref_list.append(ref)
        alt_list.append(elem)
ref_alt_df['ID'] = id_list
ref_alt_df['REF'] = ref_list
ref_alt_df['ALT'] = alt_list
ref_alt_df
outpath = 'results/variant_control_map/variant_control_ref_alt_%s.tsv'%('C_positive_heart_CAD')
ref_alt_df.to_csv(outpath, sep='\t', index=False)

#### GC_Selvarajan variant negative control 364 (try to generalize)
- GC_Selvarajan:REF_rs4848980|STARR-seq-HepG2_fwd_tile1-1	GC_Selvarajan:ALT_rs4848980|STARR-seq-HepG2_fwd_tile1-1_rs4848980 
- Merged headers: GC_Selvarajan:REF_rs9660819|STARR-seq-HepG2~rs9661525|STARR-seq-HepG2_fwd_tile1-1


In [45]:
# how many alt in set of variant controls

selvarajan = variant_control_vars[variant_control_vars['info'] == 'GC_Selvarajan']
selvarajan_alt = selvarajan[selvarajan['allele'] == 'alt']
selvarajan_alt # => 199 but in subsequent table are only 198 entries (Hypothesis: are duplicated alts in there? OR One alternative does not have a reference)

# duplicates:
selvarajan_alt['name'].duplicated().sum()

# check in ref_dict number of alternative
alt_count = 0
for ref, alt in ref_dict.items():
    alt_count += len(alt)
print(alt_count) # 198

# concat all alternative from ref_dict and check which is missing from selvarajan_alt list
all_alt = []
for alt in ref_dict.values():
    all_alt += alt

all_alt_in_df = selvarajan_alt['name'].to_list()
all_alt_in_df_set = set(all_alt_in_df)
print(len(all_alt_in_df_set))
all_matched_alt_set = set(all_alt)
print(len(all_matched_alt_set))

# symmetric_difference
all_alt_in_df_set.symmetric_difference(all_matched_alt_set)

198
199
198


AttributeError: 'list' object has no attribute 'symmetric_difference'

In [31]:
from collections import defaultdict

# for group in ['C_positive_heart_CAD', 'GC_Selvarajan']:
for group in ['GC_Selvarajan']:
    group_variants = variant_control_vars[variant_control_vars['info'] == group]
    header_list = group_variants['name'].to_list()

    # Go over each REF: put sequence into dict and if rsID is matching (rsID=ALT_<rsID>) then put into dict and compare
    # each ref should have at least 1 element in dict
    header_list.sort(reverse=True) # first REF then ALT (headers are not different until "ALT_" or "REF_")
    ref_dict = defaultdict(list)
    for header in header_list:
        if 'REF_' in header: # refs need to be unique
            # check tiling
            ref_unique_element = get_matching_element_between_ref_alt(header)
            for alt_header in header_list:
                if 'ALT_' in alt_header:
                    alt_unique_element = get_matching_element_between_ref_alt(alt_header)
                    if ref_unique_element == alt_unique_element:
                        ref_dict[header].append(alt_header)
        # break
    # print(ref_dict)

    # # investigate number of elements in ref_dict values
    for ref, alt in ref_dict.items():
        if len(alt) != 1:
            # throw error
            print('WARNING: more than one alt for ref')
            print(ref)

    # prepare dataframe with ref and alt from ref_dict
    ref_alt_df = pd.DataFrame(columns=['ID', 'REF', 'ALT'])
    id_list = []
    ref_list = []
    alt_list = []
    for ref, alt in ref_dict.items():
        alt_count = 0
        print(ref)
        print(alt)
        for elem in alt:
            # if len(alt) > 1: headers are merged => we need different unique ids => need a function which get the alt_counts rsID from the ref
            unique_id = get_rsID_from_GC_Selvarajan(ref, alt_count)
            unique_header = '%s:%s'%(get_label(ref), unique_id)
            id_list.append(unique_header)
            ref_list.append(ref)
            alt_list.append(elem)
            alt_count += 1
    ref_alt_df['ID'] = id_list
    ref_alt_df['REF'] = ref_list
    ref_alt_df['ALT'] = alt_list
    ref_alt_df
    # outpath = 'results/variant_control_map/variant_control_ref_alt_%s.tsv.gz'%(group)
    # ref_alt_df.to_csv(outpath, sep='\t', index=False, compression='gzip')
    print(ref_alt_df.shape)
    outpath = 'results/variant_control_map/variant_control_ref_alt_%s.tsv'%(group)
    ref_alt_df.to_csv(outpath, sep='\t', index=False)

GC_Selvarajan:REF_rs9660819|STARR-seq-HepG2~rs9661525|STARR-seq-HepG2_fwd_tile1-1
GC_Selvarajan:REF_rs7280278|STARR-seq-HepG2~rs7282405|STARR-seq-HepG2_fwd_tile1-1
GC_Selvarajan:REF_rs7139492|STARR-seq-HepG2~rs7316984|STARR-seq-HepG2_fwd_tile2-3
GC_Selvarajan:REF_rs58217496|STARR-seq-HepG2~rs58324269|STARR-seq-HepG2~rs60089052|STARR-seq-HepG2_fwd_tile1-1
GC_Selvarajan:REF_rs494207|STARR-seq-HepG2~rs649192|STARR-seq-HepG2_fwd_tile1-1
GC_Selvarajan:REF_rs4845619|STARR-seq-HepG2~rs56383622|STARR-seq-HepG2_fwd_tile1-1
GC_Selvarajan:REF_rs4537545|STARR-seq-HepG2~rs4576655|STARR-seq-HepG2_fwd_tile2-3
GC_Selvarajan:REF_rs3781780|STARR-seq-HepG2~rs3781781|STARR-seq-HepG2_fwd_tile1-1
GC_Selvarajan:REF_rs2993510|STARR-seq-HepG2~rs72856440|STARR-seq-HepG2_fwd_tile3-3
GC_Selvarajan:REF_rs2993510|STARR-seq-HepG2~rs72856440|STARR-seq-HepG2_fwd_tile2-3
GC_Selvarajan:REF_rs2993510|STARR-seq-HepG2~rs72856440|STARR-seq-HepG2_fwd_tile1-3
GC_Selvarajan:REF_rs2820321|STARR-seq-HepG2~rs2820322|STARR-seq-Hep

#### GC_Liang  variant negative control  16
- GC_Liang:REF_rs2125358|Liang_fwd_tile1-1 GC_Liang:ALT_rs2125358|Liang_fwd_tile1-1_rs2125358

In [15]:
from collections import defaultdict
GC_Liang = variant_control_vars[variant_control_vars['info'] == 'GC_Liang']
header_list = GC_Liang['name'].to_list()
header_list

# Go over each REF: put sequence into dict and if rsID is matching (rsID=ALT_<rsID>) then put into dict and compare
# each ref should have at least 1 element in dict
header_list.sort(reverse=True) # first REF then ALT (headers are not different until "ALT_" or "REF_")
ref_dict = defaultdict(list)
for header in header_list:
    if 'REF_' in header: # refs need to be unique
        ref_unique_element = get_matching_element_between_ref_alt(header)
        for alt_header in header_list:
            if 'ALT_' in alt_header:
                alt_unique_element = get_matching_element_between_ref_alt(alt_header)
                if ref_unique_element == alt_unique_element:
                    ref_dict[header].append(alt_header)
    # break
ref_dict

# # investigate number of elements in ref_dict values
# for ref, alt in ref_dict.items():
#     if len(alt) != 1:
#         print(ref)

# prepare dataframe with ref and alt from ref_dict
ref_alt_df = pd.DataFrame(columns=['ID', 'REF', 'ALT'])
id_list = []
ref_list = []
alt_list = []
for ref, alt in ref_dict.items():
    for elem in alt:
        unique_id = get_matching_element_between_ref_alt(ref)
        unique_header = '%s:%s'%(get_label(ref), unique_id)
        id_list.append(unique_header)
        ref_list.append(ref)
        alt_list.append(elem)
ref_alt_df['ID'] = id_list
ref_alt_df['REF'] = ref_list
ref_alt_df['ALT'] = alt_list
ref_alt_df

,ID,REF,ALT
0,GC_Liang:rs2838227,GC_Liang:REF_rs2838227|Liang_fwd_tile1-1,GC_Liang:ALT_rs2838227|Liang_fwd_tile1-1_rs283...
1,GC_Liang:rs2530731,GC_Liang:REF_rs2530731|Liang_fwd_tile1-1,GC_Liang:ALT_rs2530731|Liang_fwd_tile1-1_rs253...
2,GC_Liang:rs2125358,GC_Liang:REF_rs2125358|Liang_fwd_tile1-1,GC_Liang:ALT_rs2125358|Liang_fwd_tile1-1_rs212...
3,GC_Liang:rs17882077,GC_Liang:REF_rs17882077|Liang_fwd_tile1-1,GC_Liang:ALT_rs17882077|Liang_fwd_tile1-1_rs17...
4,GC_Liang:rs17603855,GC_Liang:REF_rs17603855|Liang_fwd_tile1-1,GC_Liang:ALT_rs17603855|Liang_fwd_tile1-1_rs17...
5,GC_Liang:rs10939614,GC_Liang:REF_rs10939614|Liang_fwd_tile1-1,GC_Liang:ALT_rs10939614|Liang_fwd_tile1-1_rs10...
6,GC_Liang:rs10502466,GC_Liang:REF_rs10502466|Liang_fwd_tile1-1,GC_Liang:ALT_rs10502466|Liang_fwd_tile1-1_rs10...
7,GC_Liang:rs1036014,GC_Liang:REF_rs1036014|Liang_fwd_tile1-1,GC_Liang:ALT_rs1036014|Liang_fwd_tile1-1_rs103...


#### C_positive_neuron_CD   variant positive control     91
- according to mohan not really variant controls
- C_positive_neuron_CD:p1_rs7115714_G_A_ref_50_A::chr11:120424017-120424287-mean_ratio2.22  C_positive_neuron_CD:p1_rs7115714_G_A_alt_50_A::chr11:120424017-120424287-mean_ratio2.13

In [16]:
from collections import defaultdict

for group in ['']
    C_positive_neuron_CD = variant_control_vars[variant_control_vars['info'] == 'C_positive_neuron_CD']
    header_list = C_positive_neuron_CD['name'].to_list()
    header_list

    # Go over each REF: put sequence into dict and if rsID is matching (rsID=ALT_<rsID>) then put into dict and compare
    # each ref should have at least 1 element in dict
    header_list.sort(reverse=True) # first REF then ALT (headers are not different until "ALT_" or "REF_")
    ref_dict = defaultdict(list)
    for header in header_list:
        if '_ref_' in header: # refs need to be unique
            ref_unique_element = get_matching_element_between_ref_alt(header)
            # print(header)
            for alt_header in header_list:
                if '_alt_' in alt_header:
                    alt_unique_element = get_matching_element_between_ref_alt(alt_header)
                    # print(alt_unique_element)
                    if ref_unique_element == alt_unique_element:
                        ref_dict[header].append(alt_header)
                        # print("-------- FOUND ---------")
        # break
    ref_dict

    # # investigate number of elements in ref_dict values
    # for ref, alt in ref_dict.items():
    #     if len(alt) != 1:
    #         print(ref)

    # prepare dataframe with ref and alt from ref_dict
    ref_alt_df = pd.DataFrame(columns=['ID', 'REF', 'ALT'])
    id_list = []
    ref_list = []
    alt_list = []
    for ref, alt in ref_dict.items():
        for elem in alt:
            unique_id = get_matching_element_between_ref_alt(ref)
            unique_header = '%s:%s'%(get_label(ref), unique_id)
            id_list.append(unique_header)
            ref_list.append(ref)
            alt_list.append(elem)
    ref_alt_df['ID'] = id_list
    ref_alt_df['REF'] = ref_list
    ref_alt_df['ALT'] = alt_list
    ref_alt_df

    outpath = 'results/variant_control_map/variant_control_ref_alt_%s.tsv'%(group)
    ref_alt_df.to_csv(outpath, sep='\t', index=False)

SyntaxError: invalid syntax (605436688.py, line 3)

In [5]:
pre_metadata_df_final

,name,sequence,category,class,source,ref_sequence,chrom,chrom_start,chrom_end,variant_class,variant_pos,SPDI,allele,info
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,NA,NA,NA,NA,cardiac_neuro_cava_random
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,NA,NA,NA,NA,cardiac_neuro_cava_random
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,NA,NA,NA,NA,cardiac_neuro_cava_random
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,NA,NA,NA,NA,cardiac_neuro_cava_random
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,NA,NA,NA,NA,cardiac_neuro_cava_random
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,scrambled,element inactive control,NA,GRCh38,NA,NA,NA,NA,NA,NA,NA,MK
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,scrambled,element inactive control,NA,GRCh38,NA,NA,NA,NA,NA,NA,NA,MK
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,scrambled,element inactive control,NA,GRCh38,NA,NA,NA,NA,NA,NA,NA,MK
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,scrambled,element inactive control,NA,GRCh38,NA,NA,NA,NA,NA,NA,NA,MK


In [21]:
pre_metadata_df['info'].value_counts()

info
cardiac_neuro_cava_random                                                                  73940
MK                                                                                          2397
C_positive_heart_AB                                                                          909
GC_Selvarajan                                                                                364
GC_Vista                                                                                     256
C_negative_heart_MK                                                                          243
C_negative_neuron_MK                                                                         222
C_negative_neuron_NP                                                                         217
GC_Mendelian_variants                                                                        209
GC_Kircher                                                                                   203
C_SLEA                   

In [148]:
# test category and variant_class
pre_metadata_df[pre_metadata_df['category'] == 'variant']

,header,sequence,id,category,class,source,ref_sequence,chrom,chrom_start,chrom_end,variant_class,variant_pos,SPDI,allele,info
8900,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,AGGACCGGATCAACTGTCCCAGCTCCCCACTGATGTGAAAGGTGGT...,oligo_17b5d1f079f92f165ae6b760da616314,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,ref,NA
8901,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,AGGACCGGATCAACTCCTGATCTGCCCTGTCCGTGACGCTTCTGCT...,oligo_2e53f6e8c61ef7cd84a0c218519435a9,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,ref,NA
8902,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,AGGACCGGATCAACTCCTCTGGGTGACCCGGAGAACACCAAGGCTG...,oligo_fa6b9ab66a2c9d8a83e0b807968d16de,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,ref,NA
8903,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,AGGACCGGATCAACTCCATGCGGTGGCCACAGCCTCGGGTGAGTTC...,oligo_9be9e91086b215fc8a407ca5505bf01d,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,ref,NA
8904,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,AGGACCGGATCAACTGGACTCCGGTGCCTTCGCATTCCCGAGCTGT...,oligo_512f507b4c554969462b4ce5da071afb,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,ref,NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73935,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTGGAGCTCTGCCTCACCCCACCTGGCCCCAAT...,oligo_6e74a25bd02b34708ccbf8093673ef62,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,alt,NA
73936,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTATGTCTGAATTCACCTCCAAATAATGGGAAA...,oligo_87305b21de48ce90a3caf03cd0265009,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,alt,NA
73937,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTCCTCTGCCCTCCCTGGCTTCTTCCCCTGTCC...,oligo_07249343e8d909bfd63abcc7ff28b50d,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,alt,NA
73938,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTCCTCTGCCCTCCCTGGCTTCTTCCCCTGTCC...,oligo_e35878121d9479168994f561fc4a4028,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,alt,NA


#### Correlation between chengyu and log2ratios


### Add knowledge on controls


In [22]:
# checking manually: found many ALT_REF pairs
# - Weird: GC_Vista have ";" do not follow similar pattern (no ALT_ or REF_)
pre_metadata_df.loc[:,'tmp_label'] = pre_metadata_df['header'].apply(get_label)


# check which controls are element and variant 

# check which controlls are positive and negative (positive: for neuro positive, negative: for neuro negative)

# store headers to a tsv file:
header_path = 'controll_header.tsv'

controls = pre_metadata_df[pre_metadata_df['tmp_label'] != 'cardiac_neuro_cava_random']
controls['header'].to_csv(header_path, sep='\t', index=False)

In [9]:
controls['header']

73940    GC_Atrial_fib:rs7795510|CAV1|STARR-seq-AF~rs78...
73941    GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...
73942    GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...
73943    GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...
73944    GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...
                               ...                        
80210    MK:tile_2240|chr1-116244322+116244591|scramble...
80211    MK:tile_6675|chr11-2374617+2374886|scramble_ne...
80212    MK:tile_18415|chr17-71181691+71181960|scramble...
80213    MK:tile_14356|chr15-67031618+67031887|scramble...
80214    MK:tile_22033|chr2-71506006+71506275|scramble_...
Name: header, Length: 6275, dtype: object

In [10]:
controls['tmp_label'].value_counts()


tmp_label
MK                                   2397
C_positive_heart_AB                   909
GC_Selvarajan                         364
GC_Vista                              256
C_negative_heart_MK                   243
C_negative_neuron_MK                  222
C_negative_neuron_NP                  217
GC_Mendelian_variants                 209
GC_Kircher                            203
C_SLEA                                200
GC_Cort_Chengyu                       185
C_positive_neuron_NP                   99
C_positive_heart_CAD                   97
C_positive_heart_MK                    97
C_positive_neuron_MK                   96
C_positive_neuron_CD                   94
GC_GABA_Chengyu                        85
GC_DNase_positive_shuffeled            55
GC_Atrial_fib                          45
GC_DNase_positive                      41
GC_Glut_Chengyu                        40
GC_Mohlke                              34
GC_DNase_negative_blood_shuffeled      19
GC_Liang                

In [11]:
def ref_or_alt_in_header(header):
    """Checks if "ref_" or "_alt" is in the header"""
    if 'ref_' in header.lower() or 'alt_' in header.lower():
        return True
    return False

def set_negative_class(header):
    if ref_or_alt_in_header(header):
        return 'variant negative control'
    return 'element inactive control'

def set_positive_class(header):
    if ref_or_alt_in_header(header):
        return 'variant positive control'
    return 'element active control'

#### Plan:
0. Look only on controls
1. get the positive control groups and the synthetic groups
2. iterate all groups (except for the positive control groups) (because you want to skipp the positive control groups)
  - when their headers have "alt_" or "ref_" included: 
    - 'variant negative control'
  - otherwise: 
    - 'element inactive control'

In [16]:
# get all unique values of the tmp_label column 
all_control_groups = controls['tmp_label'].unique()
# todo: add additional information; 
# todo: add label to source
### please put in the positive list the controls you want to be positive controls
positive_control_groups = ['C_positive_neuron_NP', 'C_positive_neuron_MK', 'C_positive_neuron_CD']
synthetic_control_groups = ['C_SLEA']
additional_information_groups = [group for group in all_control_groups if "dnase" in group.lower()]
additional_information_groups
negative_control_groups = [group for group in all_control_groups if not group in positive_control_groups]
negative_control_groups


['GC_Atrial_fib',
 'GC_Liang',
 'GC_Selvarajan',
 'GC_Mohlke',
 'GC_Kircher',
 'GC_Mendelian_variants',
 'C_positive_heart_CAD',
 'GC_Cort_Chengyu',
 'GC_GABA_Chengyu',
 'GC_Glut_Chengyu',
 'GC_Hon',
 'GC_Vista',
 'GC_DNase_positive',
 'GC_DNase_negative_brain',
 'GC_DNase_negative_blood',
 'C_negative_heart_MK',
 'C_negative_neuron_MK',
 'C_negative_neuron_NP',
 'C_positive_heart_MK',
 'C_positive_heart_AB',
 'C_SLEA',
 'GC_DNase_positive_shuffeled',
 'GC_DNase_negative_brain_shuffeled',
 'GC_DNase_negative_blood_shuffeled',
 'MK']

In [13]:
group_count = 0

updated_neg_control = pd.DataFrame(columns=["header", "class"])
length_counter = 0
for group in negative_control_groups:
    print(group)
    group_control = controls[controls['tmp_label'] == group]
    # check if some MK_control header have REF or ALT included included
    group_control.loc[:,'class'] = group_control['header'].apply(set_negative_class)
    group_control_sub = group_control[['header', 'class']]
    # print(group_control_sub['class'].value_counts())
    # join this information to the controls pandas dataframe
    updated_neg_control = pd.concat([updated_neg_control, group_control_sub], ignore_index=True, sort=False) 
    length_counter += group_control_sub.shape[0]
    # print(updated_neg_control['class'].value_counts())


updated_neg_control['class'].value_counts()



GC_Atrial_fib
GC_Liang
GC_Selvarajan
GC_Mohlke
GC_Kircher
GC_Mendelian_variants
C_positive_heart_CAD
GC_Cort_Chengyu
GC_GABA_Chengyu
GC_Glut_Chengyu
GC_Hon
GC_Vista
GC_DNase_positive
GC_DNase_negative_brain
GC_DNase_negative_blood
C_negative_heart_MK
C_negative_neuron_MK
C_negative_neuron_NP
C_positive_heart_MK
C_positive_heart_AB
GC_DNase_positive_shuffeled
GC_DNase_negative_brain_shuffeled
GC_DNase_negative_blood_shuffeled
MK


class
element inactive control    4819
variant negative control     967
Name: count, dtype: int64

In [14]:
print('In this dataset are %s negative controls'%(updated_neg_control['header'].shape[0])) # 5986

In this dataset are 5786 negative controls


#### Do same with positive controls

In [163]:

updated_pos_control = pd.DataFrame(columns=["header", "class"])
length_counter = 0
for group in positive_control_groups:
    print(group)
    group_control = controls[controls['tmp_label'] == group]
    # check if some MK_control header have REF or ALT included included
    group_control.loc[:,'class'] = group_control['header'].apply(set_positive_class)
    group_control_sub = group_control[['header', 'class']]
    # print(group_control_sub['class'].value_counts())
    # join this information to the controls pandas dataframe
    updated_pos_control = pd.concat([updated_pos_control, group_control_sub], ignore_index=True, sort=False) 
    length_counter += group_control_sub.shape[0]
    # print(updated_pos_control['class'].value_counts())


updated_pos_control['class'].value_counts()

C_positive_neuron_NP
C_positive_neuron_MK
C_positive_neuron_CD


class
element active control      198
variant positive control     91
Name: count, dtype: int64

In [164]:
length_counter

289

In [165]:
updated_pos_control

,header,class
0,C_positive_neuron_NP:GW18_PFC_ABC_chr5_6873074...,element active control
1,C_positive_neuron_NP:GW18_PFC_ABC_chr5_6873083...,element active control
2,C_positive_neuron_NP:NGN2_iPSC_ABC_chrX_743252...,element active control
3,C_positive_neuron_NP:GW18_PFC_ABC_NGN2_iPSC_AB...,element active control
4,C_positive_neuron_NP:NGN2_iPSC_ABC_chr10_88387...,element active control
...,...,...
284,C_positive_neuron_CD:p1_rs8049948_A_G_ref_50_A...,variant positive control
285,C_positive_neuron_CD:p1_rs6791336_T_C_ref_50_T...,variant positive control
286,C_positive_neuron_CD:p1_rs4982404_C_T_ref_50_C...,variant positive control
287,C_positive_neuron_CD:p2_rs9926320_G_A_ref_25_A...,variant positive control


In [166]:
# merge with metadata df 
pre_metadata_df_test = pre_metadata_df.merge(updated_neg_control, on='header', how='left')

pre_metadata_df_test['class'] = pre_metadata_df_test['class_y'].fillna(pre_metadata_df_test['class_x'])
pre_metadata_df_test = pre_metadata_df_test.drop(['class_x', 'class_y'], axis=1)


pre_metadata_df_test = pre_metadata_df_test.merge(updated_pos_control, on='header', how='left')
pre_metadata_df_test['class'] = pre_metadata_df_test['class_y'].fillna(pre_metadata_df_test['class_x'])
pre_metadata_df_test = pre_metadata_df_test.drop(['class_x', 'class_y'], axis=1)

pre_metadata_df_test['class'].value_counts()

class
test                        73940
element inactive control     5019
variant negative control      967
element active control        198
variant positive control       91
Name: count, dtype: int64

In [167]:
# check SLEA controlls
pre_metadata_df_test[pre_metadata_df_test['tmp_label'] == 'C_SLEA']

,header,sequence,id,category,source,ref_sequence,chrom,chrom_start,chrom_end,variant_class,variant_pos,SPDI,allele,info,tmp_label,class
77528,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,AGGACCGGATCAACTTAGGCTTCTCAAAAGTTATTTTTAAAGACTG...,oligo_e2a545a9d984399d75f1829e31a1a8b6,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,C_SLEA,element inactive control
77529,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,AGGACCGGATCAACTTAGGCTTCTCAAAAGTTATTTTTAAAGACTG...,oligo_efa28f0df5e5feed32a5fcdf423e9f0b,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,C_SLEA,element inactive control
77530,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,AGGACCGGATCAACTTAGGCTTCTCACCCCCTGACCTTTGCCCCCT...,oligo_d901ad4b07a7c993cb92b33f3f267a64,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,C_SLEA,element inactive control
77531,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,AGGACCGGATCAACTTAGGCTTCTCATGTTTGCTTTGTAACAAAAT...,oligo_7bd5f3c9c84842756445d26d59214da4,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,C_SLEA,element inactive control
77532,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,AGGACCGGATCAACTTAGGCTTCTCAAAGGTCCAGTTTGGGGATCG...,oligo_52e5854cec759d31e5c2dcd1d01e76a4,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,C_SLEA,element inactive control
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77723,C_SLEA:SLEA_hg18:chr9:82902419-82902586|6:V_Rx...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGGGCCGTGACCCCGT...,oligo_97208d39b8763704a889856eb6bc5b9b,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,C_SLEA,element inactive control
77724,C_SLEA:SLEA_hg18:chr9:82902419-82902586|7:V_AH...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGCGGGGATCGCGTGC...,oligo_1b6d517fd936db6af97c8b06328f191e,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,C_SLEA,element inactive control
77725,C_SLEA:SLEA_hg18:chr9:82902419-82902586|80:V_H...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGCCAGGCAAGAAGTG...,oligo_6377a5a1eab86d0c18d6e3f3267a949d,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,C_SLEA,element inactive control
77726,C_SLEA:SLEA_hg18:chr9:82902419-82902586|8:V_HN...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGCCAAGGTCCAGGTG...,oligo_49c1874afef2c81f8e56fe36231a1f65,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,C_SLEA,element inactive control


In [1]:


def get_control_category_from_class(row):
    """Set category of controls from class"""
    if row['tmp_label'] == "cardiac_neuro_cava_random":
        return row['catgegory']
    
    category_conversion_dict = {
        'variant positive control': 'variant', 
        'variant negative control': 'variant', 
        'element active control': 'element', 
        'element inactive control': 'element'
    }
    
    # C_SLEA
    if "C_SLEA" in row['tmp_label']:
        return 'synthetic'
    
    # scramble in name
    if "scramble" in row['header']:
        return 'scrambled'
    
    return category_conversion_dict[row['class']]

In [144]:
# set category
pre_metadata_df_test.apply(set_control_category_from_class)

,header,sequence,id,category,source,ref_sequence,chrom,chrom_start,chrom_end,variant_class,variant_pos,SPDI,allele,info,tmp_label,class
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,oligo_c32acb98ad2a851ab46621b1c3af8b44,element,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,NA,NA,NA,NA,NA,cardiac_neuro_cava_random,test
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,oligo_8deb96fab75f4f41dbc05b2d163ad89b,element,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,NA,NA,NA,NA,NA,cardiac_neuro_cava_random,test
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,oligo_d08942ae2ac12327ebaad04b395b7dc5,element,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,NA,NA,NA,NA,NA,cardiac_neuro_cava_random,test
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,oligo_d0ac046887b1f97d9c494e6a2bae69f7,element,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,NA,NA,NA,NA,NA,cardiac_neuro_cava_random,test
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,oligo_328edd51262a4c1c0ea79c9043176d20,element,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,NA,NA,NA,NA,NA,cardiac_neuro_cava_random,test
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,oligo_23474e71ce5c02846892d2e5f40bdfc4,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,MK,element inactive control
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,oligo_7cf2325b7ef95359164092b47379b3d5,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,MK,element inactive control
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,oligo_db734997542b4f69c38d4d3c6e88603e,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,MK,element inactive control
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,oligo_095855115099aa9e709dc0452fbc4edd,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,MK,element inactive control


##### Set negative controls to 

#### positive controls

## Start creating the Variant region lists:

In [5]:
# is there a header without "tile"? => No
design_df[(design_df['header'].str.contains("cardiac_neuro_cava_random") == True) & (design_df['header'].str.contains('tile') == False)]

,header,sequence,label


In [6]:
# filter for the rows with "cardiac_neuro_cava_random" in label column
cardiac_neuro_cava_random = design_df[design_df['label'] == 'cardiac_neuro_cava_random']

# do all of these rows have 2 pipes in the header?
cardiac_neuro_cava_random['header'].str.count(r'\|').value_counts()


# header
# 5     45082
# 2     27141
# 8       596
# 7       471
# 4       341
# 10      30


header
5     45082
2     27141
8       596
7       471
4       341
10      309
Name: count, dtype: int64

#### 2 pipe example
- 18340 REF
- 27141 all (if no REF_ or ALT_ then it is a region)
- example 
    - cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778476_fwd_tile1-1
    - cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778534_fwd_tile1-1
- format: <label> : <sequence_type> _ <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info>
- final columns: ['header', 'sequence', 'label', 'tile_info', 'strand', 'ensembl_id', 'enhancer_id', 'category', 'allele', 'gene_name']
- pattern = r'(?P<label>.*):(?P<allel>[A-Z]*_)?(?P<gene_name>.+)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>[a-z]+\d+[-]+\d+)'



In [17]:
# show me an example with threshold pipes in the header
num_pipes = 2
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]['header'].tolist()[0]
# 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778476_fwd_tile1-1' # gene name | ensembl id | enhancer/encodeID_tile id

# does a row with 2 pipes in the header have ALT in the header?
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]['header'].str.contains('ALT_').value_counts() # no alt there
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]['header'].str.contains('REF_').value_counts() # no ref there # 18340 REF

# investigate the rows with 2 pipes in the header and ALT in the header
card_cava_2_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]
print(card_cava_2_pipes.shape) # (27141, 3)
# card_cava_2_pipes['header'].str.contains('ALT')]['header'].tolist()[0]
card_cava_2_pipes[card_cava_2_pipes['header'].str.contains('REF_')]['header'].tolist()[10] # 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778534_fwd_tile1-1'

(27141, 3)


'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778534_fwd_tile1-1'

##### split the dataframe 2 pipe 


In [18]:
# 1: pre_header column: header without the label
# 2: tile info column: tile info (split by "_" and take the last element) + remove this from the pre_header column
# 3: strand column: strand info (split by "_" and take again the last element) + remove this from the pre_header column
# 4: gene_name + ensembl_id + enhancer_id column: split the pre_header column by "|" and take the first 3 elements (expand = True)
# format: <label> : <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info>

# short way using regex
pattern = r'(?P<label>.*):(?P<allel>[A-Z]*_)?(?P<gene_name>.+)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>[a-z]+\d+[-]+\d+)'
df_extracted = card_cava_2_pipes['header'].str.extract(pattern)
df_extracted
# concatenate df_extracted and card_cava_2_pipes
card_cava_2_pipes = pd.concat([card_cava_2_pipes, df_extracted], axis = 1)
card_cava_2_pipes

# long way: splitting manually
# # split the header column by ":" and take the second element
# card_cava_2_pipes['pre_header'] = card_cava_2_pipes['header'].str.split(':').str[1:].str.join(':')
# # tile info: split the pre_header column by "_" and take the last element
# card_cava_2_pipes['tile_info'] = card_cava_2_pipes['pre_header'].str.split('_').str[-1]
# # remove the tile info from the pre_header column
# card_cava_2_pipes['pre_header'] = card_cava_2_pipes['pre_header'].str.split('_').str[:-1].str.join('_')
# # strand info: split the pre_header column by "_" and take the last element
# card_cava_2_pipes['strand'] = card_cava_2_pipes['pre_header'].str.split('_').str[-1]
# # remove the strand info from the pre_header column
# card_cava_2_pipes['pre_header'] = card_cava_2_pipes['pre_header'].str.split('_').str[:-1].str.join('_')
# # gene_name + ensembl_id + enhancer_id: split the pre_header column by "|" and take the first 3 elements (expand = True)
# card_cava_2_pipes[['pre_gene_name', 'ensembl_id', 'enhancer_id']] = card_cava_2_pipes['pre_header'].str.split('|', expand = True)
# # remove the pre_header column
# card_cava_2_pipes.drop(columns = ['pre_header'], inplace = True)
# # category: element or variant
# card_cava_2_pipes['category'] = card_cava_2_pipes['pre_gene_name'].apply(lambda x: 'element' if '_' not in x else 'variant')
# # put alt or ref if "_"
# card_cava_2_pipes['allele'] = card_cava_2_pipes['pre_gene_name'].apply(lambda x: 'NA' if '_' not in x else x.split('_')[0].lower())
# # add gene_name column: split the pre_gene_name column by "_" and take the second element but only if "_" exists in the pre_gene_name column
# card_cava_2_pipes['gene_name'] = card_cava_2_pipes['pre_gene_name'].apply(lambda x: x.split('_')[1] if '_' in x else x)
# # drop the pre_gene_name column
# card_cava_2_pipes.drop(columns = ['pre_gene_name'], inplace = True)

# print(card_cava_2_pipes.columns)
# # ['header', 'sequence', 'label', 'tile_info', 'strand', 'ensembl_id', 'enhancer_id', 'category', 'allele', 'gene_name']
card_cava_2_pipes

,header,sequence,label,label,allel,gene_name,ensembl_id,enhancer_id,fwd_rev,tile_info
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,NaN,SKI,ENSG00000157933.11,EH38E2778476,fwd,tile1-1
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,NaN,SKI,ENSG00000157933.11,EH38E2778477,fwd,tile1-1
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,NaN,SKI,ENSG00000157933.11,EH38E2778478,fwd,tile1-1
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,NaN,SKI,ENSG00000157933.11,EH38E2778480,fwd,tile1-1
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,NaN,SKI,ENSG00000157933.11,EH38E2778484,fwd,tile1-1
...,...,...,...,...,...,...,...,...,...,...
27477,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,AGGACCGGATCAACTACCCCACTGCTGCACCAGATTGAGCTGGAGA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,REF_,G6PD,ENSG00000160211.20,EH38E3949715,rev,tile1-1
27478,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,AGGACCGGATCAACTTTGCTGAGTAGTATCCGTTGTATGAATGCAC...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,REF_,G6PD,ENSG00000160211.20,EH38E3949725,rev,tile1-1
27479,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,AGGACCGGATCAACTGGAGCTCTGCCTCACCCCACCTGGCCCCAAT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,REF_,G6PD,ENSG00000160211.20,EH38E3949733,rev,tile1-1
27480,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,AGGACCGGATCAACTATGTCTGAATTCACCTCCAAATAATGGGAAA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,REF_,G6PD,ENSG00000160211.20,EH38E2774396,rev,tile1-1


#### 4 pipe example (what group is this, mohan?)
- 242 REF / 99 (region)
- example: 
    - cardiac_neuro_cava_random:CSDE1|ENSG00000009307.17|EH38E1378377~NRAS|ENSG00000213281.5|EH38E1378377_rev_tile1-1
    - cardiac_neuro_cava_random:REF_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1
- format: `<label> : [<sequence_type>_]*<gene name> | <ensembl id> | <enhancer/encode id> ~ <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info>`
- assumption reference is also category variant
- final columns: ['header', 'sequence', 'label', 'additional_gene_association',
       'tile_info', 'strand', 'ensembl_id', 'enhancer_id', 'category',
       'allele', 'gene_name'] => additional gene association

In [13]:
num_pipes = 4
# Examples for the header string
# cardiac_neuro_cava_random:CSDE1|ENSG00000009307.17|EH38E1378377~NRAS|ENSG00000213281.5|EH38E1378377_rev_tile1-1
# cardiac_neuro_cava_random:REF_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1
# format: `<label> : [<allele>_]*<gene name> | <ensembl id> | <enhancer/encode id> ~ <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info>`
# Problem: allele is somethimes given (e.g. REF_ or "") but not everytime 
# solution regex: 
# pattern = r'(?P<label>.*):(?P<allel>[A-Z]*_)?(?P<gene_name>.+)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*?)_(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)\|(?P<chrom2>.*?)-(?P<pos2>.*?)-(?P<ref2>.*?)-(?P<alt2>.*)'
pattern = r'(?P<label>.*):(?P<allel>[A-Z]*_)?(?P<gene_name>.+)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)~(?P<gene_name2>.+)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*)'
card_cava_4_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]
# Extract values into new columns
df_extracted = card_cava_4_pipes['header'].str.extract(pattern)
df_extracted.head()

341


allel
REF_    242
Name: count, dtype: int64

In [14]:
# show me an example with threshold pipes in the header
num_pipes = 4
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]['header'].tolist()[0]
# 'cardiac_neuro_cava_random:CSDE1|ENSG00000009307.17|EH38E1378377~NRAS|ENSG00000213281.5|EH38E1378377_rev_tile1-1'

card_cava_4_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes] # 341 rows
print(len(card_cava_4_pipes)) # 341
# do all these rows have ALT in the header?
card_cava_4_pipes['header'].str.contains('ALT_').value_counts() # no, no alt 
card_cava_4_pipes['header'].str.contains('REF_').value_counts() # True, 242 

# check the header of something containing REF_
card_cava_4_pipes[card_cava_4_pipes['header'].str.contains('REF_')]['header'].tolist()[0] # 'cardiac_neuro_cava_random:REF_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1'

# check the header of something not containing REF_
card_cava_4_pipes[~card_cava_4_pipes['header'].str.contains('REF_')]['header'].tolist()[0] # 'cardiac_neuro_cava_random:CSDE1|ENSG00000009307.17|EH38E1378377~NRAS|ENSG00000213281.5|EH38E1378377_rev_tile1-1'

# check how many rows have ~ in the header
card_cava_4_pipes['header'].str.count('~').value_counts() # all have just one "~" 341

# check if all rows have 3 "_" in the header
card_cava_4_pipes['header'].str.count('_').value_counts() # all have 3 "_" 341

341


header
6    242
5     99
Name: count, dtype: int64

In [15]:
# format: `<label> : [<sequence_type>_]*<gene name> | <ensembl id> | <enhancer/encode id> ~ <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info>`
# 1: pre_header column: header without the label
# 2: "additional_gene_association" column: split the pre_header column by "~" and take the second element
# 3: tile info column: tile info (split "additional_gene_association" by "_" and take the last element) + remove this from the column
# 4: strand info column: strand info (split "additional_gene_association" by "_" and take again the last element) + remove this from the column
# 5: remove the "additional_gene_association" column from the pre_header column (split by "~" and take the first element)
# 6: pre_gene_name + ensembl_id + enhancer_id: split the pre_header column by "|" and take the first 3 elements (expand = True)
# 7: remove the pre_header column
# 8: sequence_type: put "region" if pre_gene_name does not contain "_" otherwise split the pre_gene_name column by "_" and take the first element

# fast way using regex: 
pattern = r'(?P<label>.*):(?P<allel>[A-Z]*_)?(?P<gene_name>.+)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)~(?P<gene_name2>.+)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*)'
card_cava_4_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]
df_extracted = card_cava_4_pipes['header'].str.extract(pattern)
print(df_extracted.head())

# # Concatenate the extracted columns with the original DataFrame
card_cava_4_pipes = pd.concat([card_cava_4_pipes, df_extracted], axis=1)
print(card_cava_4_pipes.head())

# long way: splitting manually
# # split the header column by ":" and take the second element
# card_cava_4_pipes['pre_header'] = card_cava_4_pipes['header'].str.split(':').str[1:].str.join(':')
# # additional_gene_association: split the pre_header column by "~" and take the second element
# card_cava_4_pipes['additional_gene_association'] = card_cava_4_pipes['pre_header'].str.split('~').str[1]
# # tile info: split the additional_gene_association column by "_" and take the last element
# card_cava_4_pipes['tile_info'] = card_cava_4_pipes['additional_gene_association'].str.split('_').str[-1]
# # remove the tile info from the additional_gene_association column
# card_cava_4_pipes['additional_gene_association'] = card_cava_4_pipes['additional_gene_association'].str.split('_').str[:-1].str.join('_')
# # strand info: split the additional_gene_association column by "_" and take the last element
# card_cava_4_pipes['strand'] = card_cava_4_pipes['additional_gene_association'].str.split('_').str[-1]
# # remove the strand info from the additional_gene_association column
# card_cava_4_pipes['additional_gene_association'] = card_cava_4_pipes['additional_gene_association'].str.split('_').str[:-1].str.join('_')
# # remove the additional_gene_association column from the pre_header column
# card_cava_4_pipes['pre_header'] = card_cava_4_pipes['pre_header'].str.split('~').str[0]
# # pre_gene_name + ensembl_id + enhancer_id: split the pre_header column by "|" and take the first 3 elements (expand = True)
# card_cava_4_pipes[['pre_gene_name', 'ensembl_id', 'enhancer_id']] = card_cava_4_pipes['pre_header'].str.split('|', expand = True)
# # remove the pre_header column
# card_cava_4_pipes.drop(columns = ['pre_header'], inplace = True)
# # category: element or variant
# card_cava_4_pipes['category'] = card_cava_4_pipes['pre_gene_name'].apply(lambda x: 'element' if '_' not in x else 'variant')
# # sequence_type: put "region" if pre_gene_name does not contain "_" otherwise split the pre_gene_name column by "_" and take the first element
# card_cava_4_pipes['allele'] = card_cava_4_pipes['pre_gene_name'].apply(lambda x: 'NA' if '_' not in x else x.split('_')[0].lower())

# # add gene_name column: split the pre_gene_name column by "_" and take the second element but only if "_" exists in the pre_gene_name column
# card_cava_4_pipes['gene_name'] = card_cava_4_pipes['pre_gene_name'].apply(lambda x: x.split('_')[1] if '_' in x else x)
# # drop the pre_gene_name column
# card_cava_4_pipes.drop(columns = ['pre_gene_name'], inplace = True)

# print(card_cava_4_pipes.columns)
# # ['header', 'sequence', 'label', 'additional_gene_association', 'tile_info', 'strand', 'ensembl_id', 'enhancer_id', 'category', 'allele', 'gene_name']
# card_cava_4_pipes


                         label allel gene_name          ensembl_id  \
708  cardiac_neuro_cava_random   NaN     CSDE1  ENSG00000009307.17   
709  cardiac_neuro_cava_random   NaN     CSDE1  ENSG00000009307.17   
710  cardiac_neuro_cava_random   NaN     CSDE1  ENSG00000009307.17   
711  cardiac_neuro_cava_random   NaN     CSDE1  ENSG00000009307.17   
712  cardiac_neuro_cava_random   NaN     CSDE1  ENSG00000009307.17   

      enhancer_id gene_name2        ensembl_id2  enhancer_id2 fwd_rev  \
708  EH38E1378377       NRAS  ENSG00000213281.5  EH38E1378377     rev   
709  EH38E2832502       NRAS  ENSG00000213281.5  EH38E2832502     rev   
710  EH38E2832508       NRAS  ENSG00000213281.5  EH38E2832508     rev   
711  EH38E2832513       NRAS  ENSG00000213281.5  EH38E2832513     rev   
712  EH38E1378387       NRAS  ENSG00000213281.5  EH38E1378387     rev   

    tile_info  
708   tile1-1  
709   tile1-1  
710   tile1-1  
711   tile1-1  
712   tile1-1  
                                            

enhancer id column; other genes with same enhancer
- mohan idea: 2D array or 3D array (gene column will have multiple genes)
- EH38E1378377~NRAS|ensemble id ... 
- fwd: (+ strand)
- rev: (- strand)
- gene name does not implicitly tell you the enhancer id (e.g. JUP|ENSG00000173801.17|EH38E3223379)

#### 5 pipe examples are all alt / variants (45082)
- 45082 alt
- example: 
    - cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778471|1-2179591-T-C
    - cardiac_neuro_cava_random:ALT_ST3GAL3|ENSG00000126091.21|EH38E1342813_fwd_tile1-1_ST3GAL3|ENSG00000126091.21|EH38E1342813|1-43835787-G-A
    - cardiac_neuro_cava_random:ALT_IGLV3-25|ENSG00000211659.2|EH38E3470175_fwd_tile1-1_IGLV3-25|ENSG00000211659.2|EH38E3470175|22-22639049-T-C
- format: `<label> : <sequence_type (all ALT)> _ <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info> _ <gene name> | <ensembl id> | <enhancer/encode id> | <variant info>`
- pattern = r'(?P<label>.*):ALT_(?P<gene_name>.*?)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*?)_(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)\|(?P<chrom2>.*?)-(?P<pos2>.*?)-(?P<ref2>.*?)-(?P<alt2>.*)'

In [43]:
num_pipes = 5
# show me an example with 5 pipes in the header
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]['header'].tolist()[2323]
# idx 0: cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778471|1-2179591-T-C
# idx 10: cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778513_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778513|1-2203222-G-A
# idx 24: cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778546_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778546|1-2215527-T-C
# idx 2400: cardiac_neuro_cava_random:ALT_POMGNT1|ENSG00000085998.15|EH38E2809115_rev_tile1-1_POMGNT1|ENSG00000085998.15|EH38E2809115|1-46191391-T-C
# idx 2323: cardiac_neuro_cava_random:ALT_ST3GAL3|ENSG00000126091.21|EH38E1342813_fwd_tile1-1_ST3GAL3|ENSG00000126091.21|EH38E1342813|1-43835787-G-A


# investigate and find pattern
card_cava_5_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes] 
# does all of these rows have ALT_ after the first ":" in the header?
card_cava_5_pipes['header'].str.split(':').str[1].str.contains('ALT_').value_counts() # yes => all with 5 pipes have ALT_ in name (45082)
# card_cava_5_pipes['header'].str.split(':').str[1].str.contains('REF_').value_counts() #

# # do all of these rows end with the regex /-[A-Z]*-[A-Z]*/
# card_cava_5_pipes['header'].str.extract(r'(-[A-Z]*-[A-Z]*)$')[0].value_counts() # yes => all with 5 pipes have ALT_ in name

# # check if all rows have the same number of "-"
# card_cava_5_pipes['header'].str.count('-').value_counts() # 44237 have 4 and 845 have 6
# # check what the 6 "-" rows look like
# card_cava_5_pipes[card_cava_5_pipes['header'].str.count('-') == 6]['header'].tolist()[0] # cardiac_neuro_cava_random:ALT_IGLV3-25|ENSG00000211659.2|EH38E3470175_fwd_tile1-1_IGLV3-25|ENSG00000211659.2|EH38E3470175|22-22639049-T-C

# # investigate the different number of "-" cases
# print("header with 4 '-': ", card_cava_5_pipes[card_cava_5_pipes['header'].str.count(r'-') == 4]['header'].to_list()[4:8]) 
# print("header with 6 '-': ", card_cava_5_pipes[card_cava_5_pipes['header'].str.count(r'-') == 6]['header'].to_list()[10:14]) # some genes do have "-" in there name

# do all of them have 3 "_" between 2 and 3
# list(card_cava_5_pipes['header'].str.split('|'))


header
True    45082
Name: count, dtype: int64

In [33]:
# format: `<label> : <sequence_type (all ALT)> _ <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info> _ <gene name> | <ensembl id> | <enhancer/encode id> | <variant info>`
# 1: pre_header column: header without the label
# 2: "additional_gene_association" column: split the pre_header column by "~" and take the second element
# 3: tile info column: tile info (split "additional_gene_association" by "_" and take the last element) + remove this from the column
# 4: strand info column: strand info (split "additional_gene_association" by "_" and take again the last element) + remove this from the column
# 5: remove the "additional_gene_association" column from the pre_header column (split by "~" and take the first element)
# 6: pre_gene_name + ensembl_id + enhancer_id: split the pre_header column by "|" and take the first 3 elements (expand = True)
# 7: remove the pre_header column
# 8: sequence_type: put "region" if pre_gene_name does not contain "_" otherwise split the pre_gene_name column by "_" and take the first element

# Define the regular expression pattern
pattern = r'(?P<label>.*):ALT_(?P<gene_name>.*?)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*?)_(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)\|(?P<chrom2>.*?)-(?P<pos2>.*?)-(?P<ref2>.*?)-(?P<alt2>.*)'

# Extract values into new columns
df_extracted = card_cava_5_pipes['header'].str.extract(pattern)

# # Concatenate the extracted columns with the original DataFrame
card_cava_5_pipes = pd.concat([card_cava_5_pipes, df_extracted], axis=1)
print(card_cava_5_pipes.head())
# gene_name == gene_name2
## checking if same gene name or enhancer or ensembl id for all sequences => no, but for the majority
#! Might be interesting for the resulting table
# df_extracted["same_gene_names"] = df_extracted["gene_name"] == df_extracted["gene_name2"]
# print(len(df_extracted) - df_extracted["same_gene_names"].sum())
# df_extracted["same_enhancer"] = df_extracted["enhancer_id"] == df_extracted["enhancer_id2"] 
# print(len(df_extracted) - df_extracted["same_enhancer"].sum())
# df_extracted["same_ensemble"] = df_extracted["ensembl_id"] == df_extracted["ensembl_id2"]
# print(len(df_extracted) - df_extracted["same_ensemble"].sum())
# df_extracted[df_extracted["gene_name"] != df_extracted["gene_name2"]]

,label,gene_name,ensembl_id,enhancer_id,fwd_rev,tile_info,gene_name2,ensembl_id2,enhancer_id2,chrom2,pos2,ref2,alt2
27482,cardiac_neuro_cava_random,SKI,ENSG00000157933.11,EH38E2778471,fwd,tile1-1,SKI,ENSG00000157933.11,EH38E2778471,1,2179591,T,C
27483,cardiac_neuro_cava_random,SKI,ENSG00000157933.11,EH38E2778490,fwd,tile1-1,SKI,ENSG00000157933.11,EH38E2778490,1,2191444,G,A
27484,cardiac_neuro_cava_random,SKI,ENSG00000157933.11,EH38E2778492,fwd,tile1-1,SKI,ENSG00000157933.11,EH38E2778492,1,2192015,G,T
27485,cardiac_neuro_cava_random,SKI,ENSG00000157933.11,EH38E1311587,fwd,tile1-1,SKI,ENSG00000157933.11,EH38E1311587,1,2192366,T,G
27486,cardiac_neuro_cava_random,SKI,ENSG00000157933.11,EH38E2778494,fwd,tile1-1,SKI,ENSG00000157933.11,EH38E2778494,1,2193142,G,A


#### 7 pipe examples (471) 
- all alt 471
- example: 
    - cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832509~NRAS|ENSG00000213281.5|EH38E2832509_rev_tile1-1_NRAS|ENSG00000213281.5|EH38E2832509|1-114691158-A-G
    - cardiac_neuro_cava_random:ALT_DEAF1|ENSG00000177030.19|EH38E2937979~SLC25A22|ENSG00000177542.11|EH38E2937979_rev_tile1-1_SLC25A22|ENSG00000177542.11|EH38E2937979|11-745581-A-G
- "~" differenciates same region for different gene
- "_" differenciates same region for region and variant id
- variant is indicated by chr-pos-ref-alt

- format: `<label> : <sequence_type (all ALT)> _ <gene name> | <ensembl id> | <enhancer/encode id> ~ <gene name (additional gene association) | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info> _ <gene name> | <ensembl id> | <enhancer/encode id> | <variant info>`

- `pattern = r'(?P<label>.*):ALT_(?P<gene_name>.*?)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)~(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*?)_(?P<gene_name3>.*?)\|(?P<ensembl_id3>.*?)\|(?P<enhancer_id3>.*?)\|(?P<chrom3>.*?)-(?P<pos3>.*?)-(?P<ref3>.*?)-(?P<alt3>.*)'` 

In [31]:
num_pipes = 7
# show me an example with num_pipes pipes in the header
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]['header'].tolist()[23]
# 'cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832509~NRAS|ENSG00000213281.5|EH38E2832509_rev_tile1-1_NRAS|ENSG00000213281.5|EH38E2832509|1-114691158-A-G'
# 'cardiac_neuro_cava_random:ALT_DEAF1|ENSG00000177030.19|EH38E2937979~SLC25A22|ENSG00000177542.11|EH38E2937979_rev_tile1-1_SLC25A22|ENSG00000177542.11|EH38E2937979|11-745581-A-G'


# # investigate and find pattern
card_cava_7_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes] 
# # does all of these rows have ALT_ after the first ":" in the header?
# card_cava_7_pipes['header'].str.split(':').str[1].str.startswith('ALT_').value_counts() # 471
# # card_cava_7_pipes['header'].str.split(':').str[1].str.startswith('REF_').value_counts() # no ref

# does each row has one ~
print("Check if all have one ~:", sum(card_cava_7_pipes['header'].str.count('~') == 1)) # 471 all have one ~ between 2 and third "|"
print("Check if all have one ~ between second and third '|': ", sum(card_cava_7_pipes['header'].str.split(r'\|').str[2].str.count('~') == 1)) # 471 all have one ~ between 2 and third "|"

# how many contain "rev" and how many contain "fwd"
print("All headers with rev: ", sum(card_cava_7_pipes['header'].str.count('rev'))) # 392


# Define the regular expression pattern
pattern = r'(?P<label>.*):ALT_(?P<gene_name>.*?)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)~(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*?)_(?P<gene_name3>.*?)\|(?P<ensembl_id3>.*?)\|(?P<enhancer_id3>.*?)\|(?P<chrom3>.*?)-(?P<pos3>.*?)-(?P<ref3>.*?)-(?P<alt3>.*)'

# Extract values into new columns
df_extracted = card_cava_7_pipes['header'].str.extract(pattern)
df_extracted.head()


Check if all have one ~: 471
Check if all have one ~ between second and third '|':  471
All headers with rev:  392


,label,gene_name,ensembl_id,enhancer_id,gene_name2,ensembl_id2,enhancer_id2,fwd_rev,tile_info,gene_name3,ensembl_id3,enhancer_id3,chrom3,pos3,ref3,alt3
30703,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832509,NRAS,ENSG00000213281.5,EH38E2832509,rev,tile1-1,NRAS,ENSG00000213281.5,EH38E2832509,1,114691158,A,G
30705,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832510,NRAS,ENSG00000213281.5,EH38E2832510,rev,tile1-1,NRAS,ENSG00000213281.5,EH38E2832510,1,114691688,G,A
30706,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832510,NRAS,ENSG00000213281.5,EH38E2832510,rev,tile1-1,CSDE1,ENSG00000009307.17,EH38E2832510,1,114691802,G,C
30707,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832511,NRAS,ENSG00000213281.5,EH38E2832511,rev,tile1-1,NRAS,ENSG00000213281.5,EH38E2832511,1,114692408,G,A
30708,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832511,NRAS,ENSG00000213281.5,EH38E2832511,rev,tile1-1,NRAS,ENSG00000213281.5,EH38E2832511,1,114692413,C,T


#### 8 pipe example 596
- 596 alt
-'cardiac_neuro_cava_random:ALT_DRD4|ENSG00000069696.7|EH38E2937745_fwd_tile1-1_DEAF1|ENSG00000177030.19|EH38E2937745|11-596480-T-C~DRD4|ENSG00000069696.7|EH38E2937745|11-596480-T-C'
- 
- pattern = r'(?P<label>.*):ALT_(?P<gene_name>.*?)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*?)_(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)\|(?P<chrom2>.*?)-(?P<pos2>.*?)-(?P<ref2>.*?)-(?P<alt2>.*)~(?P<gene_name3>.*?)\|(?P<ensembl_id3>.*?)\|(?P<enhancer_id3>.*?)\|(?P<chrom3>.*?)-(?P<pos3>.*?)-(?P<ref3>.*?)-(?P<alt3>.*)'


In [22]:
num_pipes = 8
# show me an example with num_pipes pipes in the header
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]['header'].tolist()[200]
# 'cardiac_neuro_cava_random:ALT_DRD4|ENSG00000069696.7|EH38E2937745_fwd_tile1-1_DEAF1|ENSG00000177030.19|EH38E2937745|11-596480-T-C~DRD4|ENSG00000069696.7|EH38E2937745|11-596480-T-C'

# investigate and find pattern
card_cava_8_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes] 
# # does all of these rows have ALT_ after the first ":" in the header?
# card_cava_8_pipes['header'].str.split(':').str[1].str.startswith('ALT_').value_counts() # 596
# # card_cava_8_pipes['header'].str.split(':').str[1].str.startswith('REF_').value_counts() # no ref

# check if all have one ~
print("Check if all have one ~:", sum(card_cava_8_pipes['header'].str.count('~') == 1)) # 596 all have one ~ between 2 and third "|"
print("Check if all have one ~ between 5 and 6 '|': ", sum(card_cava_8_pipes['header'].str.split(r'\|').str[5].str.count('~') == 1)) # 596 all have one ~ 
# check if all have same number of "_"
print("Check if all have three _ between 2 and 3 '|': ", sum(card_cava_8_pipes['header'].str.split(r'\|').str[2].str.count('_') == 3))
print("Number of tile: ", sum(card_cava_8_pipes['header'].str.count('tile'))) # expected: 596 given: 596

# Define the regular expression pattern
pattern = r'(?P<label>.*):ALT_(?P<gene_name>.*?)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*?)_(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)\|(?P<chrom2>.*?)-(?P<pos2>.*?)-(?P<ref2>.*?)-(?P<alt2>.*)~(?P<gene_name3>.*?)\|(?P<ensembl_id3>.*?)\|(?P<enhancer_id3>.*?)\|(?P<chrom3>.*?)-(?P<pos3>.*?)-(?P<ref3>.*?)-(?P<alt3>.*)'
# Extract values into new columns
df_extracted = card_cava_8_pipes['header'].str.extract(pattern)
df_extracted.head()

Check if all have one ~: 596
Check if all have one ~ between 5 and 6 '|':  596
Check if all have three _ between 2 and 3 '|':  596
Number of tile:  596


,label,gene_name,ensembl_id,enhancer_id,fwd_rev,tile_info,gene_name2,ensembl_id2,enhancer_id2,chrom2,pos2,ref2,alt2,gene_name3,ensembl_id3,enhancer_id3,chrom3,pos3,ref3,alt3
35409,cardiac_neuro_cava_random,DRD4,ENSG00000069696.7,EH38E2937745,fwd,tile1-1,DEAF1,ENSG00000177030.19,EH38E2937745,11,596480,T,C,DRD4,ENSG00000069696.7,EH38E2937745,11,596480,T,C
35410,cardiac_neuro_cava_random,DEAF1,ENSG00000177030.19,EH38E2937745,rev,tile1-1,DEAF1,ENSG00000177030.19,EH38E2937745,11,596480,T,C,DRD4,ENSG00000069696.7,EH38E2937745,11,596480,T,C
35411,cardiac_neuro_cava_random,DRD4,ENSG00000069696.7,EH38E2937745,fwd,tile1-1,DEAF1,ENSG00000177030.19,EH38E2937745,11,596499,G,C,DRD4,ENSG00000069696.7,EH38E2937745,11,596499,G,C
35412,cardiac_neuro_cava_random,DEAF1,ENSG00000177030.19,EH38E2937745,rev,tile1-1,DEAF1,ENSG00000177030.19,EH38E2937745,11,596499,G,C,DRD4,ENSG00000069696.7,EH38E2937745,11,596499,G,C
35415,cardiac_neuro_cava_random,DRD4,ENSG00000069696.7,EH38E2937745,fwd,tile1-1,DEAF1,ENSG00000177030.19,EH38E2937745,11,596672,G,T,DRD4,ENSG00000069696.7,EH38E2937745,11,596672,G,T


#### 10 pipes in example 309
- 309 alt
- example: 
  - 'cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1_CSDE1|ENSG00000009307.17|EH38E2832494|1-114668983-A-G~NRAS|ENSG00000213281.5|EH38E2832494|1-114668983-A-G'
  - 'cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832521~NRAS|ENSG00000213281.5|EH38E2832521_rev_tile1-1_CSDE1|ENSG00000009307.17|EH38E2832521|1-114716340-T-C~NRAS|ENSG00000213281.5|EH38E2832521|1-114716340-T-C'
- do all have "~" between "|" 2 and 3? yes
- do all have "_" between "|" 4 and 5? yes 
- do all have "~" between "|" 7 and 8? yes
- pattern = r'(?P<label>.*):ALT_(?P<gene_name>.*?)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)~(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)_(?P<fwd_rev2>.*?)_(?P<tile_info2>.*?)_(?P<gene_name3>.*?)\|(?P<ensembl_id3>.*?)\|(?P<enhancer_id3>.*?)\|(?P<chrom3>.*?)-(?P<pos3>.*?)-(?P<ref3>.*?)-(?P<alt3>.*)~(?P<gene_name4>.*?)\|(?P<ensembl_id4>.*?)\|(?P<enhancer_id4>.*?)\|(?P<chrom4>.*?)-(?P<pos4>.*?)-(?P<ref4>.*?)-(?P<alt4>.*)'

In [30]:
num_pipes = 10
# show me an example with num_pipes pipes in the header
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]['header'].tolist()[20]
# 'cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1_CSDE1|ENSG00000009307.17|EH38E2832494|1-114668983-A-G~NRAS|ENSG00000213281.5|EH38E2832494|1-114668983-A-G'
# 'cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832521~NRAS|ENSG00000213281.5|EH38E2832521_rev_tile1-1_CSDE1|ENSG00000009307.17|EH38E2832521|1-114716340-T-C~NRAS|ENSG00000213281.5|EH38E2832521|1-114716340-T-C'

# investigate and find pattern
card_cava_10_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes] 
# does all of these rows have ALT_ after the first ":" in the header?
card_cava_10_pipes['header'].str.split(':').str[1].str.startswith('ALT_').value_counts() # 309
# card_cava_10_pipes['header'].str.split(':').str[1].str.startswith('REF_').value_counts() # no ref

# - do all have "~" between "|" 2 and 3?
print("Check if all have one ~ between 2 and 3 '|': ", sum(card_cava_10_pipes['header'].str.split(r'\|').str[2].str.count('~') == 1)) # 309 all have one ~ 

# - do all have "_" between "|" 4 and 5?
print("Check if all have three _ between 4 and 5 '|': ", sum(card_cava_10_pipes['header'].str.split(r'\|').str[4].str.count('_') == 3))

# - do all have "~" between "|" 7 and 8? 
print("Check if all have one ~ between 7 and 8 '|': ", sum(card_cava_10_pipes['header'].str.split(r'\|').str[7].str.count('~') == 1)) # 309 all have one ~ 

# define regex
pattern = r'(?P<label>.*):ALT_(?P<gene_name>.*?)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)~(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)_(?P<fwd_rev2>.*?)_(?P<tile_info2>.*?)_(?P<gene_name3>.*?)\|(?P<ensembl_id3>.*?)\|(?P<enhancer_id3>.*?)\|(?P<chrom3>.*?)-(?P<pos3>.*?)-(?P<ref3>.*?)-(?P<alt3>.*)~(?P<gene_name4>.*?)\|(?P<ensembl_id4>.*?)\|(?P<enhancer_id4>.*?)\|(?P<chrom4>.*?)-(?P<pos4>.*?)-(?P<ref4>.*?)-(?P<alt4>.*)'

# Extract values into new columns
df_extracted = card_cava_10_pipes['header'].str.extract(pattern)
df_extracted.head()

Check if all have one ~ between 2 and 3 '|':  309
Check if all have three _ between 4 and 5 '|':  309
Check if all have one ~ between 7 and 8 '|':  309


,label,gene_name,ensembl_id,enhancer_id,gene_name2,ensembl_id2,enhancer_id2,fwd_rev2,tile_info2,gene_name3,...,pos3,ref3,alt3,gene_name4,ensembl_id4,enhancer_id4,chrom4,pos4,ref4,alt4
30689,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832494,NRAS,ENSG00000213281.5,EH38E2832494,rev,tile1-1,CSDE1,...,114668983,A,G,NRAS,ENSG00000213281.5,EH38E2832494,1,114668983,A,G
30690,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E1378368,NRAS,ENSG00000213281.5,EH38E1378368,rev,tile1-1,CSDE1,...,114669283,T,C,NRAS,ENSG00000213281.5,EH38E1378368,1,114669283,T,C
30691,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832496,NRAS,ENSG00000213281.5,EH38E2832496,rev,tile1-1,CSDE1,...,114670761,G,C,NRAS,ENSG00000213281.5,EH38E2832496,1,114670761,G,C
30692,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832496,NRAS,ENSG00000213281.5,EH38E2832496,rev,tile1-1,CSDE1,...,114670764,T,C,NRAS,ENSG00000213281.5,EH38E2832496,1,114670764,T,C
30693,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832496,NRAS,ENSG00000213281.5,EH38E2832496,rev,tile1-1,CSDE1,...,114670766,G,C,NRAS,ENSG00000213281.5,EH38E2832496,1,114670766,G,C


### Conclusion:

cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778534_fwd_tile1-1
cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1_CSDE1|ENSG00000009307.17|EH38E2832494|1-114668983-A-G~NRAS|ENSG00000213281.5|EH38E2832494|1-114668983-A-G
cardiac_neuro_cava_random:ALT_DRD4|ENSG00000069696.7|EH38E2937745_fwd_tile1-1_DEAF1|ENSG00000177030.19|EH38E2937745|11-596480-T-C~DRD4|ENSG00000069696.7|EH38E2937745|11-596480-T-C
cardiac_neuro_cava_random:REF_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1
cardiac_neuro_cava_random:CSDE1|ENSG00000009307.17|EH38E1378377~NRAS|ENSG00000213281.5|EH38E1378377_rev_tile1-1

<label>:[ALT_/REF_]*<gene_name>|<ensembl_id>|<enhancer/encodeID_tile id>[][~<gene_name>|<ensembl_id>|<enhancer/encodeID_tile id>]*_<fwd/rev>_<tile_info>

### Prepare a tsv of all labels and get the number of sequences

In [18]:
# write sequences with same label in one file
# get list of all labels in dataframe
labels = design_df['label'].unique().tolist()
group_list_output_dir = config['general']['group_lists_directory']

for label_of_interest in labels: 
    # get the rows of the dataframe with the label of interest
    label_of_interest_df = design_df[design_df['label'] == label_of_interest]
    # write it to directory as tsv file
    label_of_interest_df.to_csv(os.path.join(group_list_output_dir, label_of_interest + f'_{len(label_of_interest_df)}.tsv'), sep='\t', index=False)


In [19]:
# get number of all rows with a label starting with "C_"
C_labels = design_df[design_df['label'].str.startswith('C_')]
C_labels

,header,sequence,label
74811,C_positive_heart_CAD:REF_rs17114036,AGGACCGGATCAACTAGGAAGCAGGTCATAATTAGTGATAGTCATT...,C_positive_heart_CAD
74812,C_positive_heart_CAD:REF_rs72664324,AGGACCGGATCAACTTCCTCTGCTGAACCCACAGCAATGGCAGCCG...,C_positive_heart_CAD
74813,C_positive_heart_CAD:REF_rs12740374,AGGACCGGATCAACTTGACCCAAAAGTGCTTCATTTTTCGTGCCCG...,C_positive_heart_CAD
74814,C_positive_heart_CAD:REF_rs4450010,AGGACCGGATCAACTTGAGGTCCAAGGATGTGAGAGTGACCACAGT...,C_positive_heart_CAD
74815,C_positive_heart_CAD:REF_rs34091558,AGGACCGGATCAACTCTTCTCGGCCAATGAAGGGTCAACTCCATTG...,C_positive_heart_CAD
...,...,...,...
77723,C_SLEA:SLEA_hg18:chr9:82902419-82902586|6:V_Rx...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGGGCCGTGACCCCGT...,C_SLEA
77724,C_SLEA:SLEA_hg18:chr9:82902419-82902586|7:V_AH...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGCGGGGATCGCGTGC...,C_SLEA
77725,C_SLEA:SLEA_hg18:chr9:82902419-82902586|80:V_H...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGCCAGGCAAGAAGTG...,C_SLEA
77726,C_SLEA:SLEA_hg18:chr9:82902419-82902586|8:V_HN...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGCCAAGGTCCAGGTG...,C_SLEA


## Use vcf files to get the AF of the genes
- get the vcf files to the analysis (analyze_NGN2_feather.py)

## Use sequences of the deduplicated header and look for them in the duplicated header
- if you find a match take the header and make a list

In [20]:

# load duplicate and deduplicate fasta 
design_duplicates_fasta = "/home/kisa/coding/80K_MPRA/80K-Analysis/05_variant_region_list/resources/design.fa"
design_deduplicate_fasta = "/home/kisa/coding/80K_MPRA/80K-Analysis/05_variant_region_list/resources/design_no_duplicates_sequence_and_header.fa"
# load the fasta: 
design_dup_df = create_fasta_df_from_one_line_sequence_fasta(design_duplicates_fasta)
design_dedup_df = create_fasta_df_from_one_line_sequence_fasta(design_deduplicate_fasta)

In [72]:
design_dup_df["sequence"].astype(str)[15:285]

15     AGGACCGGATCAACTGGTTTGCTGTGGCCTGGCTCGATTGAGAATC...
16     AGGACCGGATCAACTCGGGATGGTGCCCGTGGCATCTTCTGCTCGG...
17     AGGACCGGATCAACTCTTGAACTCCTGACCTTGTGAGCTACCCACC...
18     AGGACCGGATCAACTGCAGTCTGCTTTTGGGCCTGTAGATTCGTTG...
19     AGGACCGGATCAACTCCACAGGCCAGGGCACTCCCAACAGCACTGC...
                             ...                        
280    AGGACCGGATCAACTGGGTCATCCAAGGTCCCAGGATCCAGCTCAT...
281    AGGACCGGATCAACTTGTGCGTGATCACCTGTGTAGCTCCTGGAGG...
282    AGGACCGGATCAACTTAAATTAAAATAAATAAACATTTAAAAATTA...
283    AGGACCGGATCAACTCATCAGAGTAGGTGAGACCACCTAGGGAAGA...
284    AGGACCGGATCAACTAGTACTTTGTGTTCTCATGTGACAATGGGCA...
Name: sequence, Length: 270, dtype: object

In [63]:
design_dup_df["sequence"] = design_dup_df["sequence"].astype(str)[15:285]
design_dedup_df["sequence"] = design_dedup_df["sequence"].astype(str)[15:285]

# # sort both files using their sequence
design_dup_df.sort_values(by=['sequence'], inplace=True)
design_dedup_df.sort_values(by=['sequence'], inplace=True)

In [64]:
design_dup_df

,header,sequence
227,cardiac_neuro_cava_random:NOS1AP|ENSG000001989...,AGGACCGGATCAACTAAAAATTTTTAAGGGAATTTTAAGTGTGAAA...
135,cardiac_neuro_cava_random:ST3GAL3|ENSG00000126...,AGGACCGGATCAACTAAAAGAGAGACAACTACTGCTTTTACTTCAG...
211,cardiac_neuro_cava_random:NOS1AP|ENSG000001989...,AGGACCGGATCAACTAAACAGCCATTACTACTTTAATAGAGCAGAG...
265,cardiac_neuro_cava_random:PBX1|ENSG00000185630...,AGGACCGGATCAACTAAAGAAGACAGATTATTCCCCAGTTGCCAAC...
124,cardiac_neuro_cava_random:ST3GAL3|ENSG00000126...,AGGACCGGATCAACTAAAGCTCTGGGGTGGGAAAAGGGCTCCCAAG...
...,...,...
80801,MK:tile_2240|chr1-116244322+116244591|scramble...,NaN
80802,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,NaN
80803,MK:tile_18415|chr17-71181691+71181960|scramble...,NaN
80804,MK:tile_14356|chr15-67031618+67031887|scramble...,NaN


In [49]:
idx_dup = 0
idx_dedup = 0
duplicated_sequences_list = []
duplicate = False
while idx_dup < len(design_dup_df) and idx_dedup < len(design_dedup_df):
    if  design_dup_df.iloc[idx_dup]['sequence'] == design_dedup_df.iloc[idx_dedup]['sequence']:
        if (duplicate):
            # append to list
            duplicated_sequences_list[-1][2].append(design_dup_df.iloc[idx_dup]['header'])
        else:
            # create new list
            duplicated_sequences_list.append((design_dedup_df.iloc[idx_dedup]['header'], design_dedup_df.iloc[idx_dedup]['sequence'], [design_dup_df.iloc[idx_dup]['header']]))
            duplicate = True
        idx_dedup += 1
    elif design_dup_df.iloc[idx_dup]['sequence'] < design_dedup_df.iloc[idx_dedup]['sequence']:
        idx_dup += 1
        duplicate = False
    else:
        idx_dedup += 1
    
if len(design_dedup_df) != len(duplicated_sequences_list):
    for i in range(len(design_dedup_df)):
        if design_dedup_df[i] != duplicated_sequences_list[i][0]:
            print("Missing: " + str(design_dedup_df[i]))
            exit(1)

TypeError: '<' not supported between instances of 'str' and 'float'

In [43]:
duplicated_sequences_list

[('cardiac_neuro_cava_random:ALT_NCKAP1|ENSG00000061676.16|EH38E2057960_rev_tile1-1_NCKAP1|ENSG00000061676.16|EH38E2057960|2-183088247-A-T',
  'AGGACCGGATCAACTAAAAAAAAAACACAAAAAACAAAAAACAAAAAAAACCTCTTCCTTTGCTTCCATCCAAATACAGTCGTGCATCATTTAACAATGGTGATGCATTCTGAGAAATGTGTTGTTAGGCAGTTTCTTCGTGGTGTAAACATCACAGAGTGCACTCACACAAACCAAGAATGGTATGGCCTATTGCTCCAAGACTACAAACCTGTTCAGATGTTACTGTACTGACTTGTAGGCAACAGTAACACAATGGTATGTATTTGTGTATCTAAACATACACATTGCGTGAACCGA',
  ['cardiac_neuro_cava_random:ALT_NCKAP1|ENSG00000061676.16|EH38E2057960_rev_tile1-1_NCKAP1|ENSG00000061676.16|EH38E2057960|2-183088247-A-T']),
 ('cardiac_neuro_cava_random:REF_NCKAP1|ENSG00000061676.16|EH38E2057960_rev_tile1-1',
  'AGGACCGGATCAACTAAAAAAAAAACACAAAAAACAAAAAACAAAAAAAACCTCTTCCTTTGCTTCCATCCAAATACAGTCGTGCATCATTTAACAATGGTGATGCATTCTGAGAAATGTGTTGTTAGGCAGTTTCTTCGTGGTGTAAACATCACAGAGTGCACTCACACAAACCTAGAATGGTATGGCCTATTGCTCCAAGACTACAAACCTGTTCAGATGTTACTGTACTGACTTGTAGGCAACAGTAACACAATGGTATGTATTTGTGTATCTAAACATACACATTGCGTGAACCGA',
  ['cardiac_neuro

In [46]:
print(len(duplicated_sequences_list))

count_merge = 0
for tuple in duplicated_sequences_list:
    print(tuple[0])
    print(tuple[1])
    print(tuple[2])
    break
    if len(tuple[2]) >= 2:
        count_merge += 1
        
print("Mergings in the design: ", count_merge)

80215
cardiac_neuro_cava_random:ALT_NCKAP1|ENSG00000061676.16|EH38E2057960_rev_tile1-1_NCKAP1|ENSG00000061676.16|EH38E2057960|2-183088247-A-T
AGGACCGGATCAACTAAAAAAAAAACACAAAAAACAAAAAACAAAAAAAACCTCTTCCTTTGCTTCCATCCAAATACAGTCGTGCATCATTTAACAATGGTGATGCATTCTGAGAAATGTGTTGTTAGGCAGTTTCTTCGTGGTGTAAACATCACAGAGTGCACTCACACAAACCAAGAATGGTATGGCCTATTGCTCCAAGACTACAAACCTGTTCAGATGTTACTGTACTGACTTGTAGGCAACAGTAACACAATGGTATGTATTTGTGTATCTAAACATACACATTGCGTGAACCGA
['cardiac_neuro_cava_random:ALT_NCKAP1|ENSG00000061676.16|EH38E2057960_rev_tile1-1_NCKAP1|ENSG00000061676.16|EH38E2057960|2-183088247-A-T']
Mergings in the design:  0
